# D2-01 · Boxes, IoU, mAP and the region-based lineage

### Steps
1. The detection problem on Pet heads; three box formats and their conversions
2. Intersection over Union: vectorised, corner cases, how fast it falls
3. IoU-family losses (GIoU, DIoU, CIoU) and why plain IoU cannot be a loss
4. Evaluation: TP / FP by greedy matching, the precision-recall curve, AP, mAP, COCO mAP@[.5:.95]
5. History as a chain of fixes: sliding window -> R-CNN -> Fast -> Faster -> one-stage -> anchor-free -> DETR
6. The fully-convolutional trick: a classifier *is* a sliding window; Selective Search in 40 lines
7. Non-maximum suppression: greedy, Soft-NMS, class-wise; hard-negative mining

### Topics covered and learning objectives
- Represent boxes in xyxy / xywh / cxcywh, pixels or normalised, and convert without bugs
- Compute IoU for N x M boxes with broadcasting, and reason about its corner cases
- Explain why GIoU / DIoU / CIoU give gradients where IoU does not
- Compute AP and mAP exactly the way COCO / VOC do, and read a PR curve
- Place every detector of the next three notebooks on the family tree
- Remove duplicate detections with NMS / Soft-NMS and know when NMS hurts

### Time estimate
~90 min in class (guided + hands-on)

### Prerequisites
Day 1 (convnets, training, transfer learning). NumPy broadcasting.

## 0 · Setup

Mechanisms are written in NumPy (they are a few lines each, and you will reuse them in
every detector of the day); models are PyTorch. Every exercise is followed by a checker
that prints `[PASS]` or `[FAIL]` with a hint.

<details class="api-ref" data-api="np.set_printoptions,torch.manual_seed,torchkit.get_device"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.set_printoptions</code> · <code>torch.manual_seed</code> · <code>torchkit.get_device</code></summary>
<ul><li><code>np.set_printoptions(precision=2, suppress=True)</code> &nbsp; Changes how NumPy prints arrays for the rest of the session: precision=2 shows two decimals, suppress=True writes tiny numbers as 0. instead of in scientific notation (2e-05). Only the display changes, never the values.</li><li><code>torch.manual_seed(seed)</code> &nbsp; Fixes PyTorch&#x27;s random number generator, so random weights, shuffles and augmentations come out the same on every run. The notebooks call it in the setup cell and again right before creating a model, so that its initial weights are repeatable.</li><li><code>device = torchkit.get_device()</code> &nbsp; Course helper. Returns torch.device(&quot;cuda&quot;) when PyTorch can use an NVIDIA GPU, torch.device(&quot;mps&quot;) on an Apple-silicon Mac, otherwise torch.device(&quot;cpu&quot;), and prints which one it picked (for the CPU, with the number of threads PyTorch uses). Models and batches are then moved there with .to(device).</li></ul></details>

In [ ]:
import sys, pathlib, time
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:   # find the cvcourse package
    if (p / "cvcourse").is_dir():
        sys.path.insert(0, str(p)); break

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from cvcourse import viz, boxes, shapes, tests, torchkit, day2, labs

%matplotlib inline
plt.rcParams["figure.dpi"] = 90
np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(0)
device = torchkit.get_device()
GREEN, RED, GREY, BLUE, ORANGE = "#54A24B", "#E45756", "#9a9a9a", "#4C78A8", "#F58518"

## 1 · The detection problem

A classifier answers *what* is in the image. A detector answers **what, where and how
many**: its output is a *variable-length set* of `(box, class, score)` triples. That one
sentence explains everything that makes detection harder than classification:

* the output size is not fixed -> we predict a fixed, large number of candidates and filter them
  (non-maximum suppression, NMS: section 7);
* "where" needs a geometry -> boxes, and a way to compare boxes (intersection over union, IoU: section 2);
* "is this prediction right?" depends on a threshold -> precision / recall / average precision (AP, section 4)
  instead of accuracy.

Our running example is the Oxford-IIIT Pet dataset, which ships one *head* box per animal.

<details class="api-ref" data-api="torchvision.transforms.v2.ToImage"><summary><b>Functions used here</b> (click to open): <code>torchvision.transforms.v2.ToImage</code></summary>
<p><b><code>v2.ToImage()</code></b><br>Transform that turns a PIL image or an (H, W, C) NumPy array into an image tensor of shape (C, H, W). Values and dtype are kept (uint8 stays 0..255), so it is almost always followed by v2.ToDtype(torch.float32, scale=True) to get floats in [0, 1].</p><pre>&gt;&gt;&gt; from torchvision.transforms import v2
&gt;&gt;&gt; from PIL import Image
&gt;&gt;&gt; img = v2.ToImage()(Image.new(&quot;RGB&quot;, (48, 32)))       # PIL sizes are (width, height)
&gt;&gt;&gt; img.shape, img.dtype                                 # C, H, W; still uint8
(torch.Size([3, 32, 48]), torch.uint8)
&gt;&gt;&gt; v2.ToImage()(np.zeros((32, 48, 3), np.uint8)).shape  # an H, W, C array
torch.Size([3, 32, 48])</pre><p>See also: <code>torchvision.transforms.v2.ToDtype</code>, <code>torchvision.transforms.v2.Compose</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torchvision.transforms.v2.Compose,torchvision.transforms.v2.Resize,torchvision.transforms.v2.ToDtype,torchkit.get_pet_datasets,Tensor.permute,Tensor.numpy"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchvision.transforms.v2.Compose</code> · <code>torchvision.transforms.v2.Resize</code> · <code>torchvision.transforms.v2.ToDtype</code> · <code>torchkit.get_pet_datasets</code> · <code>Tensor.permute</code> · <code>Tensor.numpy</code></summary>
<ul><li><code>v2.Compose([transform1, transform2, ...])</code> &nbsp; Chains transforms into one: calling the result applies them in the listed order, each on the output of the previous one. A whole per-image pipeline (resize, augment, convert to float, normalise) becomes one callable that a Dataset applies to every sample.</li><li><code>v2.Resize(size, antialias=True)</code> &nbsp; Resizes an image tensor (C, H, W) or a batch (N, C, H, W). With one number the shorter side becomes that size and the aspect ratio is kept; with a pair (h, w) the output is exactly h x w. The dtype is kept: uint8 in, uint8 out.</li><li><code>v2.ToDtype(torch.float32, scale=True)</code> &nbsp; Converts the image to another dtype. With scale=True it also rescales the values to the range of the new type: uint8 0 .. 255 becomes float 0.0 .. 1.0. The step between the cheap uint8 transforms and Normalize.</li><li><code>train, val, classes = torchkit.get_pet_datasets(size=224, target=&quot;category&quot;, subset=None)</code> &nbsp; Course helper. Returns (train, val, classes) for Oxford-IIIT Pet (run tools/download_data.py first); images are float (3, size, size), resized and normalised with the ImageNet mean and std. target chooses the label: &quot;category&quot; gives (image, breed index 0..36) and the 37 breed names; &quot;segmentation&quot; gives (image, int64 mask (H, W)) with 0 background, 1 pet, 2 border; &quot;detection&quot; gives (image, {&quot;boxes&quot;: (N, 4) as x1, y1, x2, y2 in pixels, &quot;labels&quot;: (N,)}) for the pet&#x27;s head. subset=N keeps a class-balanced N training (and N // 4 validation) images for quick experiments.</li><li><code>x.permute(*dims)</code> &nbsp; Reorders the dimensions: the argument lists the old positions in their new order. No data is copied. Matplotlib and image files use (H, W, C); PyTorch layers use (C, H, W); permute converts.</li><li><code>x.numpy()</code> &nbsp; Returns the tensor&#x27;s data as a NumPy array without copying: both share the same memory, so changing one changes the other. Used to hand results to Matplotlib or to NumPy code.</li></ul></details>

In [ ]:
from torchvision.transforms import v2
show_tf = v2.Compose([v2.ToImage(), v2.Resize((224, 224), antialias=True), v2.ToDtype(torch.float32, scale=True)])
pet_train, pet_val, det_classes = torchkit.get_pet_datasets(target="detection", train_tf=show_tf, eval_tf=show_tf)
print(f"Pet heads: {len(pet_train)} train / {len(pet_val)} val images, classes = {det_classes}")

fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for ax, i in zip(axes.flat, range(0, 720, 60)):
    img, t = pet_val[i]
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.axis("off")
    viz.draw_boxes(ax, t["boxes"].numpy(), labels=["head"] * len(t["boxes"]), lw=2, edgecolor=ORANGE)
    ax.set_title(pet_val.image_path(i).stem.rsplit("_", 1)[0], fontsize=8)
fig.suptitle("Oxford-IIIT Pet: one head box per animal (boxes follow the resize to 224 x 224)", y=1.0)
plt.tight_layout(); plt.show()

### Three box formats

| format | numbers | used by |
|---|---|---|
| **xyxy** | $(x_1, y_1, x_2, y_2)$ corners | torchvision, PASCAL VOC xml, this course |
| **xywh** | $(x_1, y_1, w, h)$ corner + size | COCO json |
| **cxcywh** | $(c_x, c_y, w, h)$ centre + size | YOLO, DETR, anchor encodings |

Coordinates are **continuous**: x to the right, y *downwards*, area $=(x_2-x_1)(y_2-y_1)$
(the old "+1 pixel" convention is gone). Any of them can also be **normalised** by the image
width / height to $[0, 1]$, which makes them independent of the resolution (YOLO, DETR).
Mixing formats silently is the #1 detection bug, so let us draw one box in all three.

<details class="api-ref" data-api="Tensor.round"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.round</code></summary>
<ul><li><code>x.round(decimals=0)</code> &nbsp; Rounds every element to the nearest integer, or to decimals digits after the point; exact halves go to the even neighbour (0.5 -&gt; 0, 1.5 -&gt; 2). NumPy arrays have the same method: the course uses arr.round(2) to print short numbers.</li></ul></details>

In [ ]:
img, t = pet_val[300]
b = t["boxes"][0].numpy()
x1, y1, x2, y2 = b; w, h = x2 - x1, y2 - y1; cx, cy = x1 + w / 2, y1 + h / 2
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))
for ax, title in zip(axes, ["xyxy  (x1, y1, x2, y2)", "xywh  (x1, y1, w, h)", "cxcywh  (cx, cy, w, h)"]):
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.set_title(title)
    ax.add_patch(Rectangle((x1, y1), w, h, fill=False, ec=ORANGE, lw=2.5))
axes[0].plot([x1, x2], [y1, y2], "o", color=RED, ms=8)
lbl = dict(color="white", fontsize=10, weight="bold", bbox=dict(fc="black", alpha=0.6, ec="none", pad=1.5))
axes[0].annotate(f"({x1:.0f}, {y1:.0f})", (x1, y1), (x1 + 4, y1 - 6), **lbl)
axes[0].annotate(f"({x2:.0f}, {y2:.0f})", (x2, y2), (x2 - 30, y2 + 16), **lbl)
axes[1].plot([x1], [y1], "o", color=RED, ms=8)
axes[1].annotate("", (x2, y2 + 8), (x1, y2 + 8), arrowprops=dict(arrowstyle="<->", color="white", lw=2))
axes[1].text((x1 + x2) / 2, y2 + 22, f"w = {w:.0f}", ha="center", **lbl)
axes[1].annotate("", (x2 + 8, y2), (x2 + 8, y1), arrowprops=dict(arrowstyle="<->", color="white", lw=2))
axes[1].text(x2 + 12, (y1 + y2) / 2, f"h = {h:.0f}", va="center", **lbl)
axes[2].plot([cx], [cy], "+", color=RED, ms=16, mew=3)
axes[2].annotate(f"({cx:.0f}, {cy:.0f})", (cx, cy), (cx + 6, cy - 6), **lbl)
for ax in axes: ax.set_xlim(0, 224); ax.set_ylim(224, 0)
plt.tight_layout(); plt.show()
print("xyxy  :", b.round(1))
print("xywh  :", boxes.xyxy_to_xywh(b).round(1))
print("cxcywh:", boxes.xyxy_to_cxcywh(b).round(1))
print("cxcywh normalised by the 224 px image:", (boxes.xyxy_to_cxcywh(b) / 224).round(3))

### Exercise 1 · box conversions

Write the three conversions for arrays of shape `(..., 4)` (use `b[..., 0]` etc. so they
work for one box and for N boxes; `np.stack(..., -1)` puts the columns back together).

<details class="api-ref" data-api="np.stack"><summary><b>Functions you will need</b> (click to open): <code>np.stack</code></summary>
<p><b><code>np.stack(arrays, axis=0)</code></b><br>Joins arrays of the same shape along a NEW axis: k arrays of shape (N,) give (k, N) with axis=0, or (N, k) with axis=-1. The box conversions use it to put four (N,) coordinate columns back together into one (N, 4) array.</p><pre>&gt;&gt;&gt; x1, y1 = np.array([10, 30, 50]), np.array([20, 40, 60])   # one value per box, 3 boxes
&gt;&gt;&gt; np.stack([x1, y1]).shape                                  # new axis first
(2, 3)
&gt;&gt;&gt; np.stack([x1, y1], -1)                                    # new axis last: one row per box
array([[10, 20],
       [30, 40],
       [50, 60]])</pre><p><i>Watch out:</i> np.concatenate joins along an EXISTING axis ((2, 3) + (2, 3) -&gt; (4, 3)); np.stack creates a new one ((2, 3) + (2, 3) -&gt; (2, 2, 3)).</p><p>See also: <code>np.concatenate</code>, <code>torch.stack</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.asarray"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.asarray</code></summary>
<ul><li><code>np.asarray(obj)</code> &nbsp; Converts its argument (a list, a PIL image, a CPU tensor, ...) to a NumPy array; an array is returned as it is, without a copy. A grey PIL image becomes a (H, W) uint8 array, a colour one (H, W, 3).</li></ul></details>

In [ ]:
def xyxy_to_xywh(b):
    '''b: boxes (..., 4) as (x1, y1, x2, y2) -> the same boxes (..., 4) as (x1, y1, w, h).'''
    b = np.asarray(b, float)
    # 1. the four columns, one number per box each: shape (...)
    x1 = b[..., 0]
    y1 = b[..., 1]
    x2 = b[..., 2]
    y2 = b[..., 3]
    # 2. YOUR CODE: w and h, the width and the height of every box
    # 3. YOUR CODE: put the columns x1, y1, w, h back together with np.stack([...], -1) -> (..., 4); return it
    raise NotImplementedError

def xyxy_to_cxcywh(b):
    '''b: boxes (..., 4) as (x1, y1, x2, y2) -> the same boxes (..., 4) as (cx, cy, w, h).'''
    b = np.asarray(b, float)
    # 1. the four columns, one number per box each: shape (...)
    x1 = b[..., 0]
    y1 = b[..., 1]
    x2 = b[..., 2]
    y2 = b[..., 3]
    # 2. YOUR CODE: w and h, the width and the height of every box
    # 3. YOUR CODE: cx and cy, the centre: the left edge plus half the width, the top edge plus half the height
    # 4. YOUR CODE: put the columns cx, cy, w, h back together with np.stack([...], -1) -> (..., 4); return it
    raise NotImplementedError

def cxcywh_to_xyxy(b):
    '''b: boxes (..., 4) as (cx, cy, w, h) -> the same boxes (..., 4) as (x1, y1, x2, y2).'''
    b = np.asarray(b, float)
    # 1. the four columns, one number per box each: shape (...)
    cx = b[..., 0]
    cy = b[..., 1]
    w = b[..., 2]
    h = b[..., 3]
    # 2. YOUR CODE: x1 and y1, the top-left corner: the centre minus half the width / height
    # 3. YOUR CODE: x2 and y2, the bottom-right corner: the centre plus half the width / height
    # 4. YOUR CODE: put the columns x1, y1, x2, y2 back together with np.stack([...], -1) -> (..., 4); return it
    raise NotImplementedError

tests.check_box_conversions(xyxy_to_xywh, xyxy_to_cxcywh, cxcywh_to_xyxy)

> **Normalised vs pixel coordinates.** Resizing an image by a factor $s$ multiplies pixel
> boxes by $s$ but leaves normalised boxes unchanged. Libraries differ: torchvision wants
> *pixels* (xyxy), YOLO label files store *normalised* cxcywh, COCO json stores *pixel* xywh.
> Always check which one a loader returns before you train for a day on wrong boxes.

## 2 · Intersection over Union

How good is a predicted box $P$ for a ground-truth box $G$? The standard answer is

$$\mathrm{IoU}(P, G) = \frac{|P \cap G|}{|P \cup G|} = \frac{|P \cap G|}{|P| + |G| - |P \cap G|} \in [0, 1].$$

The intersection of two axis-aligned boxes is again a box: its top-left corner is the
**max** of the two top-left corners, its bottom-right the **min** of the two bottom-right
corners, and its width / height are clipped at 0 when the boxes do not overlap. IoU is
*scale invariant*: it does not care whether the boxes are 10 or 1000 pixels wide.

<details class="api-ref" data-api="np.minimum,Tensor.all"><summary><b>Functions used here</b> (click to open): <code>np.minimum</code> · <code>Tensor.all</code></summary>
<p><b><code>np.minimum(a, b)</code></b><br>Element-wise minimum of two arrays (broadcast against each other): out[i] = min(a[i], b[i]). For two xyxy boxes, the bottom-right corner of their intersection is np.minimum of the two bottom-right corners, and the top-left corner is np.maximum of the two top-left corners.</p><pre>&gt;&gt;&gt; np.minimum(np.array([1, 5, 3]), np.array([4, 2, 3]))
array([1, 2, 3])
&gt;&gt;&gt; a, b = np.array([10, 10, 50, 50]), np.array([30, 25, 70, 65])   # two xyxy boxes
&gt;&gt;&gt; np.maximum(a[:2], b[:2]), np.minimum(a[2:], b[2:])             # corners of the intersection
(array([30, 25]), array([50, 50]))</pre><p><i>Watch out:</i> np.min(a) is the smallest element of ONE array; np.minimum(a, b) compares TWO arrays element by element.</p><p>See also: <code>np.maximum</code>, <code>np.clip</code></p><p><b><code>x.all(dim=None)</code></b><br>True if every element is True (non-zero). With dim, the test runs along that dimension and gives one result per remaining position. NumPy arrays have the same method (with axis=).</p><pre>&gt;&gt;&gt; x = torch.tensor([3., 5., 1.])
&gt;&gt;&gt; (x &gt; 0).all(), (x &gt; 2).all()
(tensor(True), tensor(False))
&gt;&gt;&gt; torch.tensor([[1, 1], [0, 1]]).all(dim=1)        # one answer per row
tensor([ True, False])</pre><p>See also: <code>Tensor.any</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.array,np.maximum"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.array</code> · <code>np.maximum</code></summary>
<ul><li><code>np.array(data, dtype=None)</code> &nbsp; Creates a NumPy array from a (nested) list, copying the data. The dtype is inferred: integers give int64, any float gives float64; dtype=float gives floats from integer literals.</li><li><code>np.maximum(a, b)</code> &nbsp; The element-wise maximum of two arrays, or of an array and a number. np.maximum(values, 1e-30) replaces values below 1e-30 (including 0) so a log-scale plot does not break on log(0). Not to be confused with np.max, which returns the single largest value.</li></ul></details>

In [ ]:
cases = [("identical", [10, 10, 50, 50], [10, 10, 50, 50]),
         ("partial overlap", [10, 10, 50, 50], [30, 25, 70, 65]),
         ("touching edges", [10, 10, 40, 50], [40, 10, 70, 50]),
         ("disjoint", [5, 5, 30, 30], [45, 40, 70, 70]),
         ("one inside the other", [5, 5, 75, 75], [25, 25, 55, 55]),
         ("same centre, other shape", [25, 5, 55, 75], [5, 25, 75, 55])]
fig, axes = plt.subplots(2, 3, figsize=(12, 7.6))
for ax, (name, a, b) in zip(axes.flat, cases):
    a, b = np.array(a, float), np.array(b, float)
    ax.add_patch(Rectangle(a[:2], *(a[2:] - a[:2]), fc=BLUE, alpha=0.35, ec=BLUE, lw=2))
    ax.add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fc=ORANGE, alpha=0.35, ec=ORANGE, lw=2))
    lt, rb = np.maximum(a[:2], b[:2]), np.minimum(a[2:], b[2:])
    if (rb > lt).all():
        ax.add_patch(Rectangle(lt, *(rb - lt), fill=False, hatch="///", ec="k", lw=0))
    ax.set_xlim(0, 80); ax.set_ylim(80, 0); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{name}\nIoU = {boxes.iou_single(a, b):.3f}")
fig.suptitle("IoU corner cases (hatched = intersection)", y=1.0)
plt.tight_layout(); plt.show()

Look at three panels: *touching edges* (sharing an edge is not overlap, IoU 0), *one inside the
other* (the small box lies entirely inside the big one, yet IoU is only 0.18: IoU also counts
the area it misses) and *same centre, other shape* (perfectly centred, IoU 0.27). IoU measures
agreement of areas, not of positions.

### Lab · IoU playground
**Predict first:** two boxes of the same size overlap by half of their width. Is the IoU 0.5?
Drag the boxes to check, then do "Try this" items 1 and 5 under the lab (4 minutes). The presets
are the six cases of the figure above. Leave the GIoU / DIoU / CIoU tiles and the loss plot for
section 3, where we come back to this lab.

In [ ]:
labs.show("iou-playground")

### Exercise 2 · pairwise IoU, vectorised

`box_iou(a, b)` takes `a (N, 4)` and `b (M, 4)` in xyxy and returns the `(N, M)` matrix of
IoUs **without Python loops**: broadcast `a[:, None, :2]` against `b[None, :, :2]`. Detectors
call this on thousands of *anchors* (fixed reference boxes tiled over the image, D2-02) x tens of
boxes per image, so it must be vectorised.

<details class="api-ref" data-api="Tensor.clip"><summary><b>Functions you will need</b> (click to open): <code>Tensor.clip</code></summary>
<p><b><code>x.clip(min=None, max=None)</code></b><br>Limits every value to [min, max] (another name for clamp); with only min, values below min become min. NumPy arrays have the same method: (x2 - x1).clip(0) makes the overlap width of two boxes that do not overlap 0 instead of negative.</p><pre>&gt;&gt;&gt; x = torch.tensor([-3., 0.5, 7.])
&gt;&gt;&gt; x.clip(0)                           # lower bound only
tensor([0.0000, 0.5000, 7.0000])
&gt;&gt;&gt; x.clip(0, 1)
tensor([0.0000, 0.5000, 1.0000])
&gt;&gt;&gt; np.array([-20., 15.]).clip(0)       # the same on a NumPy array
array([ 0., 15.])</pre><p>See also: <code>Tensor.clamp</code>, <code>np.clip</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Tensor.reshape"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.reshape</code></summary>
<ul><li><code>x.reshape(*shape)</code> &nbsp; Returns the same elements in a new shape, read in row order (the last dimension changes fastest). One size may be -1: it is computed from the others. NumPy arrays have the same method.</li></ul></details>

In [ ]:
def box_area(b):
    '''b: boxes (..., 4) in xyxy -> their areas (...); a box with x2 < x1 or y2 < y1 has area 0.'''
    widths = (b[..., 2] - b[..., 0]).clip(0)       # x2 - x1, a negative width becomes 0
    heights = (b[..., 3] - b[..., 1]).clip(0)      # y2 - y1
    return widths * heights

def box_iou(a, b):
    '''a: (N, 4), b: (M, 4) boxes in xyxy -> (N, M) matrix; entry [i, j] is the IoU of a[i] and b[j].'''
    a = np.asarray(a, float).reshape(-1, 4)        # (N, 4); a single box becomes (1, 4)
    b = np.asarray(b, float).reshape(-1, 4)        # (M, 4)
    # 1. YOUR CODE: lt, the top-left corner of every intersection: the larger of the two top-left
    #               corners, for every pair at once: np.maximum of a[:, None, :2] and b[None, :, :2] -> (N, M, 2)
    # 2. YOUR CODE: rb, the bottom-right corner of every intersection: the smaller of the two
    #               bottom-right corners (a[:, None, 2:] and b[None, :, 2:]) -> (N, M, 2)
    # 3. YOUR CODE: wh, the width and height of every intersection: rb - lt, clipped at 0 -> (N, M, 2)
    # 4. YOUR CODE: inter, the intersection areas: width times height -> (N, M)
    # 5. YOUR CODE: union, area of a[i] + area of b[j] - inter, for every pair: use box_area(a)[:, None]
    #               and box_area(b)[None, :] -> (N, M)
    # 6. YOUR CODE: return inter / union (divide by np.maximum(union, 1e-12): two empty boxes give 0, not an error)
    raise NotImplementedError

def iou_of_one_pair(a, b):
    # box_iou of one box with one box is a (1, 1) matrix: return its only entry
    return box_iou(a, b)[0, 0]

tests.check_iou(iou_of_one_pair)
tests.check_box_iou_matrix(box_iou)

<details class="api-ref" data-api="torchvision.ops.box_iou,np.concatenate"><summary><b>Functions used here</b> (click to open): <code>torchvision.ops.box_iou</code> · <code>np.concatenate</code></summary>
<p><b><code>torchvision.ops.box_iou(boxes1, boxes2)</code></b><br>IoU of every box in boxes1 (N, 4) with every box in boxes2 (M, 4), returned as an (N, M) tensor. Boxes are float tensors in xyxy format (x1, y1, x2, y2). The library version of Exercise 2: same maths, runs on the GPU, used inside every torchvision detector.</p><pre>&gt;&gt;&gt; a = torch.tensor([[0., 0., 10., 10.]])               # one 10 x 10 box
&gt;&gt;&gt; b = torch.tensor([[0., 0., 10., 10.],                # the same box
...                   [5., 0., 15., 10.],                # shifted by half its width
...                   [20., 20., 30., 30.]])             # far away
&gt;&gt;&gt; torchvision.ops.box_iou(a, b)                        # (1, 3); middle: 50 / (100 + 100 - 50)
tensor([[1.0000, 0.3333, 0.0000]])</pre><p><i>Watch out:</i> It needs tensors (torch.tensor(array) first) and assumes xyxy unless you pass fmt=&quot;xywh&quot; or fmt=&quot;cxcywh&quot;: boxes in another format give wrong numbers, not an error.</p><p>See also: <code>torchvision.ops.generalized_box_iou</code>, <code>torchvision.ops.nms</code></p><p><b><code>np.concatenate(arrays, axis=0)</code></b><br>Joins arrays end to end along an EXISTING axis; all other dimensions must match. axis=0 appends rows; axis=1 puts columns side by side, e.g. top-left corners (N, 2) and bottom-right corners (N, 2) into xyxy boxes (N, 4). The AP code uses it to pad a curve with one value at each end.</p><pre>&gt;&gt;&gt; lt = np.array([[0, 0], [5, 5]]); rb = np.array([[10, 20], [8, 9]])
&gt;&gt;&gt; np.concatenate([lt, rb], 1)                     # (2, 2) and (2, 2) side by side -&gt; (2, 4)
array([[ 0,  0, 10, 20],
       [ 5,  5,  8,  9]])
&gt;&gt;&gt; np.concatenate([[0.0], np.array([0.5, 0.7]), [1.0]])
array([0. , 0.5, 0.7, 1. ])</pre><p>See also: <code>np.stack</code>, <code>torch.cat</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torch.tensor,np.random.default_rng,np.abs,Tensor.max,np.linspace"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torch.tensor</code> · <code>np.random.default_rng</code> · <code>np.abs</code> · <code>Tensor.max</code> · <code>np.linspace</code></summary>
<ul><li><code>torch.tensor(data, dtype=None, device=None)</code> &nbsp; Builds a new tensor from a Python number, a (nested) list or a NumPy array, always copying the data. The dtype is inferred: integers give int64, floats give float32; pass dtype= to choose.</li><li><code>rng = np.random.default_rng(seed)</code> &nbsp; Creates a NumPy random number generator; the same seed gives the same numbers on every run. Draw from it with rng.normal(size=...) (normal distribution), rng.random(size) (uniform in [0, 1)), rng.integers(low, high, size) or rng.permutation(n).</li><li><code>np.abs(a)</code> &nbsp; The absolute value of every element. np.abs(a - b).max() is the largest difference between two arrays: the course&#x27;s check that two implementations compute the same numbers (up to tiny rounding differences).</li><li><code>x.max() / x.max(dim)</code> &nbsp; x.max() is the largest element (a 0-dimensional tensor); x.max(dim) works along one dimension and returns two tensors, the maximum values and their positions (indices). NumPy arrays have the same method: arr.max() and arr.max(axis=...), which returns only the values.</li><li><code>np.linspace(start, stop, num=50)</code> &nbsp; num evenly spaced numbers from start to stop, both ends included. Used for plot axes and grids.</li></ul></details>

In [ ]:
# The library equivalent: torchvision.ops.box_iou (same maths, on tensors, GPU-ready)
rng = np.random.default_rng(0)
A = rng.uniform(0, 100, (1000, 2)); A = np.concatenate([A, A + rng.uniform(5, 60, (1000, 2))], 1)
G = rng.uniform(0, 100, (20, 2));   G = np.concatenate([G, G + rng.uniform(5, 60, (20, 2))], 1)
ours = boxes.box_iou(A, G)
lib = torchvision.ops.box_iou(torch.tensor(A), torch.tensor(G)).numpy()
print("IoU matrix", ours.shape, "| max |ours - torchvision| =", np.abs(ours - lib).max())

### How fast does IoU fall?

Slide a predicted box across a ground-truth box of the same size. IoU drops **much faster
than intuition suggests**: shifting a box by one third of its width already brings IoU down
to 0.5, the usual "correct detection" threshold. And because the shift is measured in pixels,
the same 4-pixel error that is harmless on a 128-pixel dog head is fatal on a 16-pixel one -
which is why COCO reports AP separately for small, medium and large objects.

In [ ]:
gt = np.array([40, 30, 100, 90], float)           # 60 x 60
shifts = np.linspace(-75, 75, 301)
ious = [boxes.iou_single(gt, gt + [s, 0, s, 0]) for s in shifts]
snap = [-70, -45, -20, 0, 20, 45]

fig = plt.figure(figsize=(14, 5.4))
for k, s in enumerate(snap):
    ax = fig.add_axes([0.03 + k * 0.16, 0.64, 0.15, 0.3])
    ax.add_patch(Rectangle(gt[:2], 60, 60, fc=GREEN, alpha=0.4, ec=GREEN, lw=2))
    ax.add_patch(Rectangle(gt[:2] + [s, 0], 60, 60, fill=False, ec=RED, lw=2.5))
    ax.set_xlim(-40, 180); ax.set_ylim(110, 10); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"shift {s:+d} px\nIoU = {boxes.iou_single(gt, gt + [s, 0, s, 0]):.2f}", fontsize=9)
ax = fig.add_axes([0.06, 0.09, 0.42, 0.45])
ax.plot(shifts, ious, color=BLUE, lw=2); ax.axhline(0.5, ls="--", color=GREY)
ax.plot(snap, [boxes.iou_single(gt, gt + [s, 0, s, 0]) for s in snap], "o", color=RED)
ax.set_xlabel("horizontal shift of the prediction (px), box width 60 px"); ax.set_ylabel("IoU"); ax.grid(alpha=.3)
ax.text(-78, 0.53, "IoU = 0.5 at shift = w/3", color="k", fontsize=9)
ax = fig.add_axes([0.56, 0.09, 0.42, 0.45])
px = np.linspace(0, 20, 81)
for size, c in zip([16, 32, 64, 128], [RED, ORANGE, GREEN, BLUE]):
    g = np.array([0, 0, size, size], float)
    ax.plot(px, [boxes.iou_single(g, g + [d, d, d, d]) for d in px], color=c, lw=2, label=f"{size} px box")
ax.axhline(0.5, ls="--", color=GREY); ax.set_xlabel("diagonal shift (px)"); ax.set_ylabel("IoU")
ax.legend(fontsize=8); ax.grid(alpha=.3); ax.set_title("same pixel error, different object sizes", fontsize=10)
plt.show()

## 3 · IoU-family losses: GIoU, DIoU, CIoU

The metric is IoU, so why not train with $\mathcal{L} = 1 - \mathrm{IoU}$? Two problems:

1. When the boxes **do not overlap**, IoU is 0 whatever the distance: the loss is flat and the
   gradient is exactly zero. The network gets no hint which way to move.
2. An L1 loss on the four corners has gradients everywhere but is **not scale invariant**
   (a 5 px error costs the same on a huge and on a tiny box) and does not match the metric.

The fixes add a penalty that keeps working without overlap. With $C$ the smallest box that
encloses both, $\rho$ the distance between centres and $c$ the diagonal of $C$:

$$\mathrm{GIoU} = \mathrm{IoU} - \frac{|C| - |P \cup G|}{|C|}, \qquad
  \mathrm{DIoU} = \mathrm{IoU} - \frac{\rho^2}{c^2}, \qquad
  \mathrm{CIoU} = \mathrm{DIoU} - \alpha v$$

where $v = \frac{4}{\pi^2}\left(\arctan\frac{w_G}{h_G} - \arctan\frac{w_P}{h_P}\right)^2$ measures the
aspect-ratio mismatch and $\alpha = \frac{v}{1 - \mathrm{IoU} + v}$ weights it (treated as a constant
when back-propagating). All three lie in $[-1, 1]$; the losses are $1 - (\cdot)$.
GIoU (2019) is in DETR; DIoU / CIoU (2020) in the YOLO family.

### Lab · IoU losses
The same lab, now started on the *disjoint* case. The arrows on the edges of the prediction P show
the negative gradient of the chosen loss, i.e. where one step of gradient descent would move each edge.
**Predict first:** for 1 - IoU there are no arrows at all. When you switch the arrows to GIoU, will
the box *slide* towards the ground truth, or *stretch* towards it? And DIoU?
Check, then do "Try this" items 2-4 (5 minutes); the lower plot is the 1-D cut of every loss.

In [ ]:
labs.show("iou-playground", preset="disjoint", loss="iou")

### Exercise 3 · GIoU

`giou(a, b)` for *aligned* pairs: `a (N, 4)` and `b (N, 4)` -> `(N,)`. Compute IoU of each pair,
then the enclosing box (min of top-left corners, max of bottom-right corners) and its area.

<details class="api-ref" data-api="Tensor.prod"><summary><b>Functions you will need</b> (click to open): <code>Tensor.prod</code></summary>
<p><b><code>x.prod(dim=None)</code></b><br>Multiplies the elements: all of them, or along dim. On an (N, 2) array of widths and heights, prod(-1) gives the N areas. NumPy arrays have the same method (with axis=).</p><pre>&gt;&gt;&gt; wh = torch.tensor([[4., 5.], [2., 0.], [3., 3.]])    # (w, h) of three boxes
&gt;&gt;&gt; wh.prod(-1)                                          # their areas
tensor([20.,  0.,  9.])</pre><p>See also: <code>Tensor.sum</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
def giou(a, b):
    '''a, b: (N, 4) boxes in xyxy, compared pair by pair (a[i] with b[i]) -> the GIoU of each pair, (N,).'''
    a = np.asarray(a, float).reshape(-1, 4)        # (N, 4)
    b = np.asarray(b, float).reshape(-1, 4)        # (N, 4)
    # 1. the IoU of each pair, as in box_iou, but pair i only with pair i (no broadcasting)
    lt = np.maximum(a[:, :2], b[:, :2])            # (N, 2) top-left corner of the intersection
    rb = np.minimum(a[:, 2:], b[:, 2:])            # (N, 2) bottom-right corner of the intersection
    wh = (rb - lt).clip(0)                         # (N, 2) its width and height, 0 without overlap
    inter = wh.prod(-1)                            # (N,) width * height
    union = box_area(a) + box_area(b) - inter      # (N,)
    iou = inter / np.maximum(union, 1e-12)         # (N,)
    # 2. YOUR CODE: c_lt, the top-left corner of the enclosing box C: the smaller of the two top-left corners -> (N, 2)
    # 3. YOUR CODE: c_rb, the bottom-right corner of C: the larger of the two bottom-right corners -> (N, 2)
    # 4. YOUR CODE: c_wh, the width and height of C (c_rb - c_lt, clipped at 0) -> (N, 2)
    # 5. YOUR CODE: c_area, the area of C: width * height -> (N,)
    # 6. YOUR CODE: empty_share, the share of C that neither box covers: (c_area - union) / c_area
    #               (divide by np.maximum(c_area, 1e-12)); return GIoU = iou - empty_share
    raise NotImplementedError

tests.check_giou(giou)

<details class="api-ref" data-api="np.repeat,np.diag"><summary><b>Functions used here</b> (click to open): <code>np.repeat</code> · <code>np.diag</code></summary>
<p><b><code>np.repeat(a, repeats, axis=None)</code></b><br>Repeats each element (with axis: each slice along that axis) `repeats` times, next to itself. D2-01 uses it to copy one GT box into N identical rows, and its method form a.repeat(3, 2) to turn a grey (H, W, 1) image into (H, W, 3).</p><pre>&gt;&gt;&gt; np.repeat(np.array([1, 2]), 3)
array([1, 1, 1, 2, 2, 2])
&gt;&gt;&gt; gt = np.array([80, 40, 120, 80])
&gt;&gt;&gt; np.repeat(gt[None], 3, 0)                  # (1, 4) -&gt; (3, 4): the same box three times
array([[ 80,  40, 120,  80],
       [ 80,  40, 120,  80],
       [ 80,  40, 120,  80]])
&gt;&gt;&gt; np.zeros((2, 2, 1)).repeat(3, 2).shape     # method form, along axis 2
(2, 2, 3)</pre><p><i>Watch out:</i> Tensor.repeat is NOT the same: torch.tensor([1, 2]).repeat(3) tiles the whole tensor, giving [1, 2, 1, 2, 1, 2]. The PyTorch twin of np.repeat is torch.repeat_interleave.</p><p>See also: <code>np.concatenate</code></p><p><b><code>np.diag(a)</code></b><br>With a 2-D array: returns its diagonal a[0, 0], a[1, 1], ... as a 1-D array. With a 1-D array: builds a square matrix with those values on the diagonal. D2-01 takes the diagonal of an (N, N) IoU matrix to get the IoU of each aligned pair (prediction i with ground truth i).</p><pre>&gt;&gt;&gt; iou = np.array([[0.9, 0.1, 0.0],
...                 [0.2, 0.7, 0.3],
...                 [0.0, 0.4, 0.5]])
&gt;&gt;&gt; np.diag(iou)                         # pair i with pair i
array([0.9, 0.7, 0.5])
&gt;&gt;&gt; np.diag([1, 2])
array([[1, 0],
       [0, 2]])</pre><p>See also: <code>Tensor.diag</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.full_like,Tensor.sum"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.full_like</code> · <code>Tensor.sum</code></summary>
<ul><li><code>np.full_like(a, fill_value)</code> &nbsp; A new array with the same shape and dtype as a, with every element set to fill_value. The course fills one with np.nan to mean &quot;not computed yet&quot;: Matplotlib&#x27;s imshow leaves NaN pixels blank.</li><li><code>x.sum(dim=None, keepdim=False)</code> &nbsp; Adds up all elements (giving a 0-dimensional tensor), or only along dim, which then disappears from the shape: for a (2, 3) tensor, sum(dim=0) adds the two rows together and gives one value per column, shape (3,). dim can be a tuple; keepdim=True keeps the summed dimension with size 1.</li></ul></details>

In [ ]:
# 1-D loss landscape: the GT is fixed, the prediction (40 x 40, then 40 x 80) slides horizontally
gt = np.array([80, 40, 120, 80], float)
xs = np.linspace(-60, 220, 400)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), sharey=True)
for ax, (pw, ph) in zip(axes, [(40, 40), (40, 80)]):
    P = np.stack([xs, np.full_like(xs, 60 - ph / 2), xs + pw, np.full_like(xs, 60 + ph / 2)], 1)
    G = np.repeat(gt[None], len(xs), 0)
    l1 = np.abs(P - G).sum(1); l1 = l1 / l1.max() * 2
    ax.plot(xs + pw / 2, l1, color=GREY, lw=2, label="L1 on corners (rescaled)")
    ax.plot(xs + pw / 2, 1 - np.diag(boxes.box_iou(P, G)), color=BLUE, lw=2.5, label="1 - IoU")
    ax.plot(xs + pw / 2, 1 - boxes.generalized_iou(P, G), color=ORANGE, lw=2, label="1 - GIoU")
    ax.plot(xs + pw / 2, 1 - boxes.distance_iou(P, G), color=GREEN, lw=2, ls="--", label="1 - DIoU")
    ax.plot(xs + pw / 2, 1 - boxes.complete_iou(P, G), color=RED, lw=1.5, ls=":", label="1 - CIoU")
    ax.axvspan(xs[0] + pw / 2, 80 - pw / 2, color=BLUE, alpha=0.06); ax.axvspan(120 + pw / 2, xs[-1] + pw / 2, color=BLUE, alpha=0.06)
    ax.text(5, 1.9, "no overlap:\n1-IoU is flat", color=BLUE, fontsize=9)
    ax.axvline(100, color="k", lw=0.8); ax.set_xlabel("x-centre of the prediction (GT centre = 100)")
    ax.set_title(f"prediction {pw} x {ph}, GT 40 x 40"); ax.grid(alpha=.3)
axes[0].set_ylabel("loss"); axes[0].legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

Read the right panel: the prediction has the wrong aspect ratio, so even perfectly centred
its IoU is 0.5. Only CIoU adds an extra penalty for that; GIoU and DIoU agree with IoU once
the boxes overlap. Now let the losses *drive* a box. We start a prediction far away from
the GT and run gradient descent on its four coordinates with each loss
(`torchvision.ops` has all of them).

<details class="api-ref" data-api="torchvision.ops.generalized_box_iou_loss,torchvision.ops.distance_box_iou_loss,torchvision.ops.complete_box_iou_loss,torch.optim.Adam,Tensor.diag,np.argmax"><summary><b>Functions used here</b> (click to open): <code>torchvision.ops.generalized_box_iou_loss</code> · <code>torchvision.ops.distance_box_iou_loss</code> · <code>torchvision.ops.complete_box_iou_loss</code> · <code>torch.optim.Adam</code> · <code>Tensor.diag</code> · <code>np.argmax</code></summary>
<p><b><code>generalized_box_iou_loss(boxes1, boxes2, reduction=&#x27;none&#x27;)</code></b><br>1 - GIoU for ALIGNED pairs: boxes1[i] against boxes2[i], both (N, 4) xyxy, giving N losses in [0, 2] (reduction=&quot;mean&quot; or &quot;sum&quot; for one number). Unlike 1 - IoU it keeps growing when boxes that do not overlap move apart, so it has a gradient. The box loss of DETR (together with L1).</p><pre>&gt;&gt;&gt; from torchvision.ops import generalized_box_iou_loss
&gt;&gt;&gt; p = torch.tensor([[0., 0., 10., 10.], [20., 0., 30., 10.]])
&gt;&gt;&gt; g = torch.tensor([[0., 0., 10., 10.], [0., 0., 10., 10.]])
&gt;&gt;&gt; generalized_box_iou_loss(p, g)       # perfect pair: 0; disjoint pair: 1 - 0 + (300 - 200) / 300
tensor([0.0000, 1.3333])</pre><p>See also: <code>torchvision.ops.generalized_box_iou</code>, <code>torchvision.ops.distance_box_iou_loss</code>, <code>torchvision.ops.complete_box_iou_loss</code></p><p><b><code>distance_box_iou_loss(boxes1, boxes2, reduction=&#x27;none&#x27;)</code></b><br>1 - DIoU for aligned pairs (N, 4) xyxy: 1 - IoU plus the squared distance between the two box centres, divided by the squared diagonal of the smallest box enclosing both. Its gradient pulls the centre of the prediction straight towards the target.</p><pre>&gt;&gt;&gt; from torchvision.ops import distance_box_iou_loss
&gt;&gt;&gt; p = torch.tensor([[20., 0., 30., 10.]]); g = torch.tensor([[0., 0., 10., 10.]])
&gt;&gt;&gt; distance_box_iou_loss(p, g)          # 1 - 0 + 20**2 / (30**2 + 10**2)
tensor([1.4000])</pre><p>See also: <code>torchvision.ops.generalized_box_iou_loss</code>, <code>torchvision.ops.complete_box_iou_loss</code></p><p><b><code>complete_box_iou_loss(boxes1, boxes2, reduction=&#x27;none&#x27;)</code></b><br>1 - CIoU for aligned pairs (N, 4) xyxy: the DIoU loss plus a term that grows when the two boxes have different aspect ratios (width / height). The box loss of the YOLOv4 / v5 generation.</p><pre>&gt;&gt;&gt; from torchvision.ops import complete_box_iou_loss, distance_box_iou_loss
&gt;&gt;&gt; p = torch.tensor([[0., 0., 10., 20.]]); g = torch.tensor([[0., 0., 10., 10.]])   # tall vs square
&gt;&gt;&gt; distance_box_iou_loss(p, g), complete_box_iou_loss(p, g)     # CIoU adds the shape mismatch
(tensor([0.5500]), tensor([0.5532]))</pre><p>See also: <code>torchvision.ops.distance_box_iou_loss</code>, <code>torchvision.ops.generalized_box_iou_loss</code></p><p><b><code>torch.optim.Adam(params, lr=0.001)</code></b><br>The Adam optimizer. Like SGD it follows the gradient, but it divides each parameter&#x27;s step by a running estimate of that gradient&#x27;s size, so a step moves about lr whatever the scale of the gradient. D2-01 uses it to move four box coordinates with the same step length for every loss; D2-04 to train free queries.</p><pre>&gt;&gt;&gt; p = torch.tensor([5.0], requires_grad=True)
&gt;&gt;&gt; opt = torch.optim.Adam([p], lr=1.0)
&gt;&gt;&gt; loss = (p ** 2).sum()                  # gradient 2 * 5 = 10
&gt;&gt;&gt; opt.zero_grad(); loss.backward(); opt.step()
&gt;&gt;&gt; p.detach()                             # moved by lr = 1 (SGD would move by lr * 10)
tensor([4.])</pre><p><i>Watch out:</i> params must be an iterable: torch.optim.Adam([p]) for single tensors, model.parameters() for a model. Adam(p) with a bare tensor raises a TypeError.</p><p>See also: <code>torch.optim.AdamW</code>, <code>torch.optim.SGD</code></p><p><b><code>x.diag()</code></b><br>The tensor version of np.diag: the diagonal of a 2-D tensor as a 1-D tensor (or a diagonal matrix from a 1-D tensor). 1 - box_iou(p, g).diag() is the IoU loss of the aligned pairs p[i], g[i].</p><pre>&gt;&gt;&gt; iou = torch.tensor([[0.9, 0.1], [0.2, 0.7]])
&gt;&gt;&gt; iou.diag()
tensor([0.9000, 0.7000])</pre><p>See also: <code>np.diag</code>, <code>torchvision.ops.box_iou</code></p><p><b><code>np.argmax(a, axis=None)</code></b><br>Index of the largest value: of the flattened array, or one per row / column along axis. On a boolean array it is the index of the first True, e.g. the first step at which IoU reached 0.5.</p><pre>&gt;&gt;&gt; print(np.argmax(np.array([0.2, 0.9, 0.4])))
1
&gt;&gt;&gt; ious = np.array([0.1, 0.3, 0.6, 0.8])
&gt;&gt;&gt; print(np.argmax(ious &gt;= 0.5))                          # the first True
2
&gt;&gt;&gt; print(np.argmax(np.array([[1, 7], [9, 2]]), axis=1))   # one per row
[1 0]</pre><p><i>Watch out:</i> If no element is True, np.argmax(mask) returns 0, as if the first one were: check mask.any() first.</p><p>See also: <code>Tensor.argmax</code>, <code>np.argsort</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Tensor.backward,Optimizer.zero_grad,Optimizer.step,Tensor.detach,Tensor.any"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.backward</code> · <code>Optimizer.zero_grad</code> · <code>Optimizer.step</code> · <code>Tensor.detach</code> · <code>Tensor.any</code></summary>
<ul><li><code>loss.backward()</code> &nbsp; Computes the gradient of a single number (usually the loss) with respect to every tensor it was computed from that has requires_grad=True, and adds it to that tensor&#x27;s .grad. One call fills the gradients of all the parameters of a model.</li><li><code>optimizer.zero_grad()</code> &nbsp; Clears the .grad of every parameter the optimiser updates (sets it to None), so that the next backward() starts from zero. Call it in every training step, before loss.backward().</li><li><code>optimizer.step()</code> &nbsp; Updates every parameter the optimiser was given, using the gradients now in .grad; for plain SGD, w &lt;- w - lr * w.grad. Call it after loss.backward().</li><li><code>x.detach()</code> &nbsp; Returns the same data as a tensor that autograd no longer tracks (requires_grad=False), without copying. Needed to turn a weight or a model output into NumPy, or to keep it without keeping its whole computation alive.</li><li><code>x.any(dim=None)</code> &nbsp; True if at least one element is True (non-zero): over the whole tensor, or along dim (one answer per row with dim=1). Top-k accuracy: compare the k guesses of each image with its label, then ask whether any of them matched.</li></ul></details>

In [ ]:
from torchvision.ops import generalized_box_iou_loss, distance_box_iou_loss, complete_box_iou_loss
def iou_loss(p, g):
    return 1 - torchvision.ops.box_iou(p, g).diag()
losses = {"1 - IoU": iou_loss, "GIoU loss": generalized_box_iou_loss,
          "DIoU loss": distance_box_iou_loss, "CIoU loss": complete_box_iou_loss}
gt_t = torch.tensor([[60., 50., 100., 110.]])
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, (name, fn) in zip(axes, losses.items()):
    p = torch.tensor([[130., 120., 160., 140.]], requires_grad=True)
    opt = torch.optim.Adam([p], lr=1.0)               # Adam: same step length for every loss
    traj = []
    for step in range(150):
        loss = fn(p, gt_t).sum()
        opt.zero_grad(); loss.backward(); opt.step()
        traj.append(p.detach().numpy()[0].copy())
    traj = np.array(traj)
    ax.add_patch(Rectangle((60, 50), 40, 60, fc=GREEN, alpha=0.35, ec=GREEN, lw=2))
    for k in [0, 20, 40, 60, 149]:
        b = traj[k]; ax.add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec=RED, alpha=0.25 + 0.75 * k / 149, lw=1.5))
    c = (traj[:, :2] + traj[:, 2:]) / 2
    ax.plot(c[:, 0], c[:, 1], ".-", color=RED, ms=3, lw=1)
    ious = boxes.box_iou(traj, gt_t.numpy())[:, 0]
    reach = f"IoU 0.5 after {np.argmax(ious >= 0.5) + 1} steps" if (ious >= 0.5).any() else "never overlaps"
    ax.set_title(f"{name}\n{reach}, final IoU {ious[-1]:.2f}", fontsize=10)
    ax.set_xlim(30, 190); ax.set_ylim(170, 20); ax.set_aspect("equal")
fig.suptitle("Gradient descent on the box coordinates, starting with no overlap (green = GT)", y=1.02)
plt.tight_layout(); plt.show()

`1 - IoU` never moves: zero gradient, the box is stuck forever. GIoU moves because it
penalises the empty part of the enclosing box: its gradient first stretches the box towards
the GT, then the IoU term takes over. DIoU pulls the **centre** straight at the target, CIoU
additionally matches the aspect ratio. (We use Adam so that every loss gets the same step
length and only the *direction* of the gradient matters.) The first step of each trajectory is
exactly the arrow field you saw in the lab for that loss. In practice detectors combine an
IoU-family loss with L1 (DETR: L1 + GIoU).

## 4 · Evaluation: precision, recall, AP, mAP

A detector outputs scored boxes. To score the detector on one class:

1. **Sort** all predictions by score, highest first.
2. Walk down the list. A prediction is a **true positive** if its best-matching *still
   unmatched* GT box has IoU $\ge$ threshold (0.5 for PASCAL VOC); that GT is then consumed.
   Otherwise it is a **false positive** (a duplicate of an already-found object is a FP!).
   GT boxes never matched are **false negatives**.
3. After each prediction $k$: $\;\text{precision}_k = \frac{TP_k}{k}$, $\;\text{recall}_k = \frac{TP_k}{\#GT}$.

Every score threshold you could choose corresponds to one point of this list, so the
**precision-recall curve** summarises all thresholds at once. **Average precision (AP)** is
the area under it (after making precision monotonically decreasing). **mAP** is the mean of
AP over classes. COCO additionally averages over IoU thresholds $0.5, 0.55, \dots, 0.95$, which
rewards *precise* boxes: that is "mAP@[.5:.95]", the number every paper reports.

In [ ]:
# A toy image: 4 GT objects and 8 predictions with hand-picked quality
rng = np.random.default_rng(7)
toy_img, toy_gt, _, _ = shapes.make_sample(np.random.default_rng(4), 160, max_objects=4, min_r=14, max_r=26)
g = toy_gt
toy_pred = np.array([g[0] + [1, -2, 2, 1],                 # good box on object 0
                     g[1] + [-2, 1, 1, 2],                 # good box on object 1
                     g[0] + [6, 5, 8, 6],                  # duplicate of object 0
                     g[2] + [2, 2, -1, -2],                # good box on object 2
                     [5, 120, 35, 150],                    # background
                     g[3] + [14, 12, 16, 14],              # badly localised object 3
                     g[3] + [-1, 1, 2, 0],                 # good box on object 3
                     [120, 5, 150, 30]], float)            # background
toy_scores = np.array([0.95, 0.91, 0.84, 0.77, 0.62, 0.55, 0.43, 0.31])
P_toy, R_toy, tp_toy = boxes.precision_recall(toy_pred, toy_scores, toy_gt, 0.5)

fig, ax = plt.subplots(1, 2, figsize=(14, 5.6), gridspec_kw=dict(width_ratios=[1, 1.15]))
ax[0].imshow(toy_img); ax[0].axis("off"); ax[0].set_title("GT (green) and 8 predictions (TP blue, FP red)")
viz.draw_boxes(ax[0], toy_gt, edgecolor=GREEN, lw=3)
for k, (b, ok) in enumerate(zip(toy_pred, tp_toy)):
    c = BLUE if ok else RED
    ax[0].add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec=c, lw=1.8, ls="-" if ok else "--"))
    ax[0].text(b[2] - 2, b[1] + 1, f"#{k + 1}", color="white", fontsize=8, ha="right", va="top", bbox=dict(fc=c, ec="none", pad=1))
ax[1].axis("off")
rows = [[f"#{k + 1}", f"{s:.2f}", f"{boxes.box_iou(b[None], toy_gt).max():.2f}", "TP" if t else "FP",
         f"{int(tp_toy[:k + 1].sum())}", f"{p:.2f}", f"{r:.2f}"] for k, (b, s, t, p, r) in enumerate(zip(toy_pred, toy_scores, tp_toy, P_toy, R_toy))]
tab = ax[1].table(cellText=rows, colLabels=["rank", "score", "best IoU", "", "cum TP", "precision", "recall"], loc="center", cellLoc="center")
tab.scale(1, 1.6); tab.set_fontsize(10)
for k, t in enumerate(tp_toy):
    tab[k + 1, 3].set_facecolor("#cfe6cf" if t else "#f6cccc")
ax[1].set_title("walk down the list sorted by score", pad=12)
plt.tight_layout(); plt.show()

Prediction #3 overlaps object 0 well, but object 0 was already claimed by #1: **duplicates
are false positives**. Prediction #6 overlaps object 3 with IoU < 0.5: a localisation error
*and* a miss at that point of the list. Now you write the matching.

### Exercise 4 · precision and recall by greedy matching

`precision_recall(pred_boxes, pred_scores, gt_boxes, iou_threshold)` takes `pred_boxes (N, 4)`,
`pred_scores (N,)` and `gt_boxes (M, 4)` and returns `(precision, recall, tp)`: three arrays of
length N in score order, where `tp` is boolean and `precision[k]`, `recall[k]` describe the top
k + 1 predictions. Use `boxes.box_iou`; mark matched GTs so they cannot be used twice (setting
their IoU to -1 is a neat trick).

<details class="api-ref" data-api="np.cumsum"><summary><b>Functions you will need</b> (click to open): <code>np.cumsum</code></summary>
<p><b><code>np.cumsum(a, axis=None)</code></b><br>Running sum: element k of the result is a[0] + ... + a[k]. On the true-positive flags of predictions sorted by score it counts the TPs among the top k + 1, the numerator of precision and recall.</p><pre>&gt;&gt;&gt; tp = np.array([True, True, False, True])
&gt;&gt;&gt; np.cumsum(tp)               # TPs so far
array([1, 2, 2, 3])
&gt;&gt;&gt; np.cumsum(~tp)              # FPs so far (~ flips True and False)
array([0, 0, 1, 1])</pre><p>See also: <code>np.maximum.accumulate</code>, <code>Tensor.cumsum</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.argsort,np.zeros,Tensor.argmax"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.argsort</code> · <code>np.zeros</code> · <code>Tensor.argmax</code></summary>
<ul><li><code>np.argsort(a)</code> &nbsp; The indices that would sort an array in ascending order; [::-1] reverses them for descending. a[np.argsort(a)] is sorted, and indexing other arrays with the same order keeps them aligned (class names, colours).</li><li><code>np.zeros(shape, dtype=float)</code> &nbsp; A new array of the given shape filled with 0.0 (float64). The standard way to allocate an output array before filling it in a loop.</li><li><code>x.argmax(dim)</code> &nbsp; The position of the largest value along dim. For logits (B, num_classes), x.argmax(dim=1) is the predicted class of every sample, shape (B,). Without dim it gives the position in the flattened tensor.</li></ul></details>

In [ ]:
def precision_recall(pred_boxes, pred_scores, gt_boxes, iou_threshold=0.5):
    '''pred_boxes (N, 4), pred_scores (N,), gt_boxes (M, 4), all xyxy ->
    (precision, recall, tp): three arrays of length N in score order (highest first);
    precision[k] and recall[k] describe the top k + 1 predictions, tp[k] is True for a true positive.'''
    # 1. the predictions in score order, highest first
    order = np.argsort(-np.asarray(pred_scores))     # (N,) indices into pred_boxes
    matched = np.zeros(len(gt_boxes), bool)          # (M,) becomes True once a GT has been found
    tp = np.zeros(len(order), bool)                  # (N,) True if the k-th prediction is a true positive
    # 2. walk down the list: the k-th best prediction is pred_boxes[i]
    for k, i in enumerate(order):
        if len(gt_boxes) == 0:
            break                                    # no GT at all: every prediction is a false positive
        # YOUR CODE: ious, the IoU of prediction i with every GT box -> (M,)
        #            (boxes.box_iou(pred_boxes[i][None], gt_boxes) is a (1, M) matrix: take its row 0)
        # YOUR CODE: set the IoU of the already matched GTs to -1, so that no GT is found twice
        # YOUR CODE: j, the index of the GT with the highest IoU
        # YOUR CODE: if that IoU is >= iou_threshold: tp[k] = True, and GT j is now matched
        raise NotImplementedError
    # 3. running counts over the list: after the top k + 1 predictions
    tp_count = np.cumsum(tp)                          # (N,) true positives so far
    fp_count = np.cumsum(~tp)                         # (N,) false positives so far
    precision = tp_count / (tp_count + fp_count)      # (N,)
    recall = tp_count / max(len(gt_boxes), 1)         # (N,)
    return precision, recall, tp

tests.check_precision_recall(precision_recall)

In [ ]:
# The PR curve, built one prediction at a time
fig, axes = plt.subplots(2, 4, figsize=(15, 7), sharex=True, sharey=True)
for k, ax in enumerate(axes.flat):
    ax.plot(R_toy[:k + 1], P_toy[:k + 1], "-", color=GREY, lw=1.5)
    ax.scatter(R_toy[:k], P_toy[:k], c=[BLUE if t else RED for t in tp_toy[:k]], s=30, alpha=0.6)
    ax.scatter([R_toy[k]], [P_toy[k]], c=[BLUE if tp_toy[k] else RED], s=140, edgecolor="k", zorder=3)
    ax.set_title(f"#{k + 1} score {toy_scores[k]:.2f}: {'TP' if tp_toy[k] else 'FP'}\nP = {P_toy[k]:.2f}, R = {R_toy[k]:.2f}", fontsize=10)
    ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.08); ax.grid(alpha=.3)
for ax in axes[1]: ax.set_xlabel("recall")
for ax in axes[:, 0]: ax.set_ylabel("precision")
fig.suptitle("A TP moves the curve right (recall up); a FP moves it down (precision down)", y=1.0)
plt.tight_layout(); plt.show()

### From the zig-zag to one number

The raw curve zig-zags. All modern protocols first replace precision at recall $r$ by the
**best precision at any recall $\ge r$** (the "envelope": you would never pick a threshold
that is worse in both precision and recall), then integrate:

$$\mathrm{AP} = \int_0^1 p_{\text{interp}}(r)\,dr, \qquad p_{\text{interp}}(r) = \max_{\tilde r \ge r} p(\tilde r).$$

VOC2007 sampled $p_{\text{interp}}$ at 11 recall points, VOC2010+ integrates exactly ("all
points"), COCO samples 101 points.

### Lab · PR curve and AP
The lab starts with the same 8 predictions and 4 objects as the figures above, already sorted by score.
**Predict first:** raising the score threshold from 0.3 to 0.9 removes all four false positives
from the reported detections. Does AP go up?
Move the threshold to check, then work through "Try this" (6 minutes).

In [ ]:
labs.show("pr-curve")

### Exercise 5 · average precision (all points)

Pad the curve with `(recall 0, precision 0)` at the start and `(recall 1, precision 0)` at the end
(as in the code), make precision monotone from the right, then sum `(r[i+1] - r[i]) * p[i+1]`
wherever recall changes.

<details class="api-ref" data-api="np.maximum.accumulate,np.where"><summary><b>Functions used here</b> (click to open): <code>np.maximum.accumulate</code> · <code>np.where</code></summary>
<p><b><code>np.maximum.accumulate(a)</code></b><br>Running maximum: element k is max(a[0], ..., a[k]). Applied to the reversed precision array and reversed back, it gives at every recall the best precision at that recall or higher: the monotone envelope that AP integrates.</p><pre>&gt;&gt;&gt; p = np.array([1.0, 0.5, 0.67, 0.75, 0.6])      # raw precision, in score order
&gt;&gt;&gt; np.maximum.accumulate(p)                        # running max from the left
array([1., 1., 1., 1., 1.])
&gt;&gt;&gt; np.maximum.accumulate(p[::-1])[::-1]            # from the right: the envelope
array([1.  , 0.75, 0.75, 0.75, 0.6 ])</pre><p>See also: <code>np.cumsum</code>, <code>np.maximum</code></p><p><b><code>np.where(condition)  or  np.where(condition, x, y)</code></b><br>With one argument: the indices where condition is True, as a tuple with one index array per dimension (take [0] for a 1-D array). With three: a new array that takes x where condition is True and y elsewhere, element by element.</p><pre>&gt;&gt;&gt; r = np.array([0.0, 0.25, 0.25, 0.5])
&gt;&gt;&gt; np.where(r[1:] != r[:-1])              # a tuple: (indices,)
(array([0, 2]),)
&gt;&gt;&gt; np.where(r[1:] != r[:-1])[0]           # where recall changes
array([0, 2])
&gt;&gt;&gt; np.where(np.array([0.2, 0.9, 0.6]) &gt; 0.5, 1, 0)
array([0, 1, 1])</pre><p><i>Watch out:</i> The one-argument form returns a tuple, not an array: forgetting the [0] gives confusing errors later.</p><p>See also: <code>torch.where</code>, <code>np.argmax</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.sum"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.sum</code></summary>
<ul><li><code>np.sum(a, axis=None)</code> &nbsp; Adds up all elements of an array (one number), or only along axis. In the convolution exercises, np.sum(patch * k) multiplies a patch with the kernel element by element and adds everything up: one output value.</li></ul></details>

In [ ]:
def average_precision(precision, recall):
    '''precision, recall: arrays of length N (from precision_recall) -> AP, the area under the PR curve (a float).'''
    # 1. pad the curve: (recall 0, precision 0) in front, (recall 1, precision 0) at the end
    p = np.concatenate([[0.0], precision, [0.0]])    # (N + 2,)
    r = np.concatenate([[0.0], recall, [1.0]])       # (N + 2,)
    # 2. YOUR CODE: the envelope: a loop from right to left, i = len(p) - 2, ..., 1, 0
    #               (range(len(p) - 2, -1, -1)), that sets p[i] = max(p[i], p[i + 1])
    # 3. YOUR CODE: the area: ap = 0.0, then a loop over i = 0, ..., len(r) - 2; wherever recall
    #               changes (r[i + 1] != r[i]), add the rectangle (r[i + 1] - r[i]) * p[i + 1]
    # 4. YOUR CODE: return ap as a float
    raise NotImplementedError

tests.check_average_precision(average_precision)

In [ ]:
p = np.concatenate([[0.0], P_toy, [0.0]]); r = np.concatenate([[0.0], R_toy, [1.0]])
env = np.maximum.accumulate(p[::-1])[::-1]                # the envelope of Exercise 5 in one NumPy call
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.fill_between(r, env, step="pre", color=BLUE, alpha=0.18, label="AP = shaded area")
ax.step(r, env, where="pre", color=BLUE, lw=2.5, label="interpolated (envelope)")
ax.plot(R_toy, P_toy, "o-", color=GREY, lw=1, label="raw precision / recall")
r11 = np.linspace(0, 1, 11)
ax.plot(r11, [env[r >= t].max() if (r >= t).any() else 0 for t in r11], "x", color=RED, ms=9, mew=2, label="VOC2007 11-point samples")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.05); ax.grid(alpha=.3); ax.legend(loc="lower left", fontsize=9)
ax.set_title("AP of the toy example")
plt.show()
for m in ["all", "interp11", "interp101"]:
    print(f"AP ({m:9s}) = {boxes.average_precision(P_toy, R_toy, m):.3f}")

This figure is the lab's starting state; `boxes.average_precision` defaults to the COCO
101-point version, your Exercise 5 is the all-points one.

### mAP over classes, IoU thresholds and object sizes

Let us evaluate a *simulated* detector on 300 shapes images: every object is found with a
few pixels of localisation noise, sometimes with the wrong class, plus some background false
positives. `boxes.mean_average_precision` pools the predictions of all images per class
(this is how VOC / COCO do it - the PR curve is per *dataset*, not per image).

<details class="api-ref" data-api="np.round,np.clip"><summary><b>Functions used here</b> (click to open): <code>np.round</code> · <code>np.clip</code></summary>
<p><b><code>np.round(a, decimals=0)</code></b><br>Rounds every element to the given number of decimals. D2-01 cleans up the IoU thresholds 0.5, 0.55, ..., 0.95 made by np.arange with it: the floating-point steps of arange are not exact.</p><pre>&gt;&gt;&gt; t = np.arange(0.5, 0.96, 0.05)
&gt;&gt;&gt; float(t[2])                     # not exactly 0.6
0.6000000000000001
&gt;&gt;&gt; np.round(t, 2)
array([0.5 , 0.55, 0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95])</pre><p><i>Watch out:</i> Halves go to the nearest EVEN number: np.round(0.5) is 0, np.round(1.5) and np.round(2.5) are both 2.</p><p>See also: <code>Tensor.round</code></p><p><b><code>np.clip(a, a_min, a_max)</code></b><br>Limits every element to [a_min, a_max]: smaller values become a_min, larger ones a_max. Keeps simulated scores inside [0, 1], and probabilities away from 0 before a log.</p><pre>&gt;&gt;&gt; np.clip(np.array([-0.2, 0.5, 1.3]), 0, 1)
array([0. , 0.5, 1. ])
&gt;&gt;&gt; np.log(np.clip(np.array([0.0, 0.5]), 1e-7, 1))      # no -inf
array([-16.11809565,  -0.69314718])</pre><p>See also: <code>Tensor.clip</code>, <code>np.minimum</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.arange,np.mean"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.arange</code> · <code>np.mean</code></summary>
<ul><li><code>np.arange(stop) / np.arange(start, stop, step, dtype=None)</code> &nbsp; Like Python&#x27;s range, as a NumPy array: from start (default 0) up to but not including stop. With reshape, the quickest way to build a small test array with distinct values.</li><li><code>np.mean(a, axis=None)</code> &nbsp; The average of an array or of a plain Python list of numbers; with axis=k only along that axis. Arrays also have it as a method: a.mean(axis=k).</li></ul></details>

In [ ]:
sim_imgs, sim_gt_boxes, sim_gt_labels, _ = shapes.make_dataset(300, 128, seed=5)
rng = np.random.default_rng(1)
sim_preds, sim_gts = [], []
for gb, gl in zip(sim_gt_boxes, sim_gt_labels):
    pb = gb + rng.normal(0, 2.5, gb.shape)                                 # ~2.5 px noise per coordinate
    pl = np.where(rng.uniform(size=len(gl)) < 0.08, rng.integers(0, 3, len(gl)), gl)   # 8% wrong class
    ps = np.clip(0.5 + 0.5 * np.diag(boxes.box_iou(pb, gb)) + rng.normal(0, 0.1, len(gl)), 0, 1)
    nf = rng.integers(0, 3)                                                # 0-2 background false positives
    fb = rng.uniform(0, 100, (nf, 2)); fb = np.concatenate([fb, fb + rng.uniform(10, 30, (nf, 2))], 1)
    sim_preds.append({"boxes": np.concatenate([pb, fb]), "scores": np.concatenate([ps, rng.uniform(0.2, 0.8, nf)]),
                      "labels": np.concatenate([pl, rng.integers(0, 3, nf)])})
    sim_gts.append({"boxes": gb, "labels": gl})

thr = np.round(np.arange(0.5, 0.96, 0.05), 2)
ap_per_thr = []                          # one array of 3 per-class APs for every IoU threshold
for iou_thr in thr:
    _, ap_per_class = boxes.mean_average_precision(sim_preds, sim_gts, 3, (iou_thr,))
    ap_per_thr.append(ap_per_class)
map50, per_class50 = boxes.mean_average_precision(sim_preds, sim_gts, 3, (0.5,))
map_coco, _ = boxes.mean_average_precision(sim_preds, sim_gts, 3, tuple(thr))

# size buckets (COCO uses 32^2 / 96^2 px on ~640 px images; we scale them to our 128 px images)
def size_filter(d, lo, hi):
    # keep only the boxes (and their scores / labels) with an area in [lo, hi)
    area = boxes.box_area(d["boxes"])
    in_bucket = (area >= lo) & (area < hi)
    filtered = {}
    for key, values in d.items():
        filtered[key] = values[in_bucket]
    return filtered
buckets = {"small (< 24^2)": (0, 24 ** 2), "medium": (24 ** 2, 36 ** 2), "large (>= 36^2)": (36 ** 2, 1e9)}
size_map = {}                            # bucket name -> mAP@[.5:.95] on the objects of that size only
for bucket_name, (lo, hi) in buckets.items():
    preds_in_bucket = []
    for pred_dict in sim_preds:
        preds_in_bucket.append(size_filter(pred_dict, lo, hi))
    gts_in_bucket = []
    for gt_dict in sim_gts:
        gts_in_bucket.append(size_filter(gt_dict, lo, hi))
    bucket_map, _ = boxes.mean_average_precision(preds_in_bucket, gts_in_bucket, 3, tuple(thr))
    size_map[bucket_name] = bucket_map

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for c, name in enumerate(shapes.CLASS_NAMES):
    axes[0].plot(thr, [a[c] for a in ap_per_thr], "o-", color=shapes.CLASS_COLORS[c], label=name)
axes[0].plot(thr, [np.mean(a) for a in ap_per_thr], "k--", lw=2, label="mAP")
axes[0].set_xlabel("IoU threshold"); axes[0].set_ylabel("AP"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)
axes[0].set_title(f"mAP@0.5 = {map50:.3f}   mAP@[.5:.95] = {map_coco:.3f}")
axes[1].bar(shapes.CLASS_NAMES, per_class50, color=shapes.CLASS_COLORS); axes[1].set_ylim(0, 1)
axes[1].set_title("AP@0.5 per class"); [axes[1].text(i, v + 0.02, f"{v:.2f}", ha="center") for i, v in enumerate(per_class50)]
axes[2].bar(list(size_map), list(size_map.values()), color=[RED, ORANGE, GREEN]); axes[2].set_ylim(0, 1)
axes[2].set_title("mAP@[.5:.95] by object size (same 2.5 px noise)")
for i, v in enumerate(size_map.values()): axes[2].text(i, v + 0.02, f"{v:.2f}", ha="center")
plt.tight_layout(); plt.show()

Three lessons in one figure: (1) with a few pixels of noise, mAP@0.5 is nearly perfect but
AP collapses at IoU 0.9 - **mAP@[.5:.95] measures localisation quality**; (2) mAP is the plain
average over classes, so a rare class counts as much as a common one; (3) the *same* pixel
noise hurts small objects most. When someone quotes "mAP", always ask: which IoU thresholds,
which dataset, which object sizes. (Library equivalent: `torchmetrics.detection.MeanAveragePrecision`
or `pycocotools`; they implement exactly this protocol.)

## 5 · Ten years of detectors as a chain of fixes

Every detector you will meet today is the previous one with **one bottleneck removed**.
Keep this map in mind; D2-02 opens the two-stage branch, D2-03 the one-stage branch,
D2-04 the transformer branch.

In [ ]:
steps = [
    ("Sliding window\n+ CNN classifier", "2013 (OverFeat)", "classify every crop at\nevery position and scale", "~10^5 crops / image"),
    ("R-CNN", "2014", "~2000 Selective-Search\nproposals, CNN on each", "~47 s / image"),
    ("Fast R-CNN", "2015", "ONE conv pass, crop\nfeatures with RoI pooling", "~2 s (proposals dominate)"),
    ("Faster R-CNN", "2015", "proposals by a learned\nRPN on the same features", "~0.2 s (5 fps)"),
    ("One-stage\nYOLO / SSD /\nRetinaNet", "2016-17", "no proposals: dense grid\npredicts boxes + classes", "20-60 fps"),
    ("Anchor-free\nFCOS / CenterNet", "2019", "no anchor boxes: points\npredict distances / peaks", "fewer hyper-parameters"),
    ("Set prediction\nDETR / RT-DETR", "2020-23", "transformer queries +\nHungarian matching,\nno NMS", "end to end"),
]
fixes = ["too many crops", "one CNN per proposal", "proposals on CPU", "two stages", "anchor tuning", "NMS + hand rules"]
fig, ax = plt.subplots(figsize=(16, 4.6)); ax.axis("off"); ax.set_xlim(0, 16); ax.set_ylim(0.3, 5.2)
cols = [GREY, "#9ecae1", "#6baed6", BLUE, ORANGE, GREEN, "#B279A2"]
for k, ((name, year, idea, speed), c) in enumerate(zip(steps, cols)):
    x = 0.15 + k * 2.27
    ax.add_patch(FancyBboxPatch((x, 1.35), 1.95, 3.2, boxstyle="round,pad=0.05", fc=c, ec="k", alpha=0.85))
    ax.text(x + 0.97, 4.25, name, ha="center", va="top", fontsize=10, weight="bold")
    ax.text(x + 0.97, 3.3, year, ha="center", fontsize=9, style="italic")
    ax.text(x + 0.97, 2.5, idea, ha="center", va="center", fontsize=8.2)
    ax.text(x + 0.97, 1.55, speed, ha="center", fontsize=8, color="#222")
    if k < len(fixes):
        ax.annotate("", (x + 2.27, 2.9), (x + 1.98, 2.9), arrowprops=dict(arrowstyle="-|>", lw=2))
        ax.text(x + 2.12, 0.95, "fixes:\n" + fixes[k], ha="center", va="top", fontsize=8, color=RED)
ax.text(8, 5.0, "Each step removes the bottleneck of the previous one (speeds are rough, different GPUs - read the trend)",
        ha="center", fontsize=11)
plt.show()

Read the red "fixes" labels under the arrows: each names the bottleneck of the box on its left that
the box on its right removed. The bottom line of every box is the pay-off (47 s per image for
R-CNN, 0.2 s for Faster R-CNN two years later).

## 6 · The fully-convolutional trick: a classifier *is* a sliding window

The naive detector (and the 2013 OverFeat paper) slides a classifier over the image. That
is thousands of forward passes - but look at what a ResNet does: convolutions, then **global
average pooling**, then a linear layer. A linear layer applied after averaging equals the
average of the linear layer applied *at every position*, and a linear layer applied at every
position is a **1x1 convolution**. So: drop the pooling, turn `fc` into a 1x1 conv, feed a
bigger image, and you get a **map of class scores** - one forward pass, a sliding window with
stride 32 for free, all computation shared between overlapping windows.

We use an ImageNet ResNet-50 (it knows ~120 dog breeds and 5 cat classes) on a canvas with
several Pet images pasted on it. (Swap in your D1-04 fine-tuned Pet model if you have it.)

<details class="api-ref" data-api="torchvision.models.resnet50,Module.children"><summary><b>Functions used here</b> (click to open): <code>torchvision.models.resnet50</code> · <code>Module.children</code></summary>
<p><b><code>torchvision.models.resnet50(weights=None)</code></b><br>Builds a ResNet-50, the 50-layer ResNet made of the bottleneck blocks of D1-03 (25.6 M parameters, 2048 features before the classifier). weights=&quot;IMAGENET1K_V2&quot; downloads the ImageNet-trained weights (about 100 MB, once); weights=None gives random ones. Input (N, 3, H, W) normalised images, output (N, 1000) ImageNet logits.</p><pre>&gt;&gt;&gt; net = torchvision.models.resnet50(weights=None).eval()      # random weights, no download
&gt;&gt;&gt; net(torch.zeros(1, 3, 224, 224)).shape
torch.Size([1, 1000])
&gt;&gt;&gt; [name for name, _ in net.named_children()][-3:]             # what comes after layer4
[&#x27;layer4&#x27;, &#x27;avgpool&#x27;, &#x27;fc&#x27;]
&gt;&gt;&gt; net.fc
Linear(in_features=2048, out_features=1000, bias=True)</pre><p>See also: <code>torchvision.models.resnet18</code>, <code>Module.children</code></p><p><b><code>model.children()</code></b><br>Iterates over the direct sub-modules of a model in the order they were defined (not recursively). list(net.children())[:-2] drops a ResNet&#x27;s last two children (average pooling and fc), and nn.Sequential(*...) glues the rest back together into a feature extractor.</p><pre>&gt;&gt;&gt; net = torchvision.models.resnet18(weights=None)
&gt;&gt;&gt; list(net.children())[-2:]
[AdaptiveAvgPool2d(output_size=(1, 1)), Linear(in_features=512, out_features=1000, bias=True)]
&gt;&gt;&gt; body = nn.Sequential(*list(net.children())[:-2])
&gt;&gt;&gt; body(torch.zeros(1, 3, 224, 224)).shape          # a 7 x 7 feature map instead of 1000 logits
torch.Size([1, 512, 7, 7])</pre><p><i>Watch out:</i> Rebuilding with nn.Sequential only works when forward() just calls the children one after the other. It is fine for ResNet up to layer4; a model that reshapes or adds branches inside forward() breaks.</p><p>See also: <code>Module.modules</code>, <code>nn.Sequential</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Module.eval,nn.Sequential,nn.Conv2d,Tensor.clone,torch.no_grad,Tensor.mean,Tensor.abs,Tensor.softmax,torch.stack"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Module.eval</code> · <code>nn.Sequential</code> · <code>nn.Conv2d</code> · <code>Tensor.clone</code> · <code>torch.no_grad</code> · <code>Tensor.mean</code> · <code>Tensor.abs</code> · <code>Tensor.softmax</code> · <code>torch.stack</code></summary>
<ul><li><code>model.eval()</code> &nbsp; Puts the model and all its layers into evaluation mode: dropout is switched off and BatchNorm uses the running averages collected during training, so a prediction no longer depends on the rest of the batch. Returns the model, so get_model(...).eval() works in one line.</li><li><code>nn.Sequential(layer1, layer2, ...)</code> &nbsp; A model made of layers applied one after the other, the output of each being the input of the next. Saves writing a class when forward() would only chain layers. The layers can be read back by position: model[0].</li><li><code>nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=True)</code> &nbsp; A convolution layer: out_channels learned kernels of size (in_channels, k, k), each slid over the input exactly like this notebook&#x27;s cross-correlation (no kernel flip). Input (B, C_in, H, W) -&gt; output (B, C_out, H_out, W_out), with H_out from the output-size formula. The weight has shape (C_out, C_in, k, k), the bias (C_out,).</li><li><code>x.clone()</code> &nbsp; A copy of the tensor with its own memory: later changes to either one do not affect the other. Plain assignment (b = a) copies nothing, it gives the same tensor a second name. w.detach().clone() is the usual way to take a snapshot of a tensor that keeps being trained.</li><li><code>with torch.no_grad(): ...</code> &nbsp; A with block in which autograd records nothing: results have requires_grad=False, and computing them takes less memory and time. Use it when you only evaluate a model (testing, plotting its predictions), never around the training step itself.</li><li><code>x.mean(dim=None, keepdim=False)</code> &nbsp; The average of all elements, or along dim (a number or a tuple), which then disappears from the shape. For a batch (B, C, H, W), mean(dim=(2, 3)) averages over the pixels and gives (B, C): global average pooling. NumPy arrays have the same method, with axis= for dim=.</li><li><code>x.abs()</code> &nbsp; The absolute value of every element. (a - b).abs().max() is the largest difference between two tensors: the usual check that two results agree.</li><li><code>logits.softmax(dim)</code> &nbsp; Turns scores (logits) into probabilities along dimension dim: exp(x) / sum(exp(x)). Every value ends up between 0 and 1 and they sum to 1 along dim; the order of the scores is kept, so the argmax does not change.</li><li><code>torch.stack(tensors, dim=0)</code> &nbsp; Joins a list of tensors of identical shape along a new dimension: N tensors of shape (C, H, W) give one tensor (N, C, H, W). This is what a DataLoader does with the items of a batch.</li></ul></details>

In [ ]:
from PIL import Image
net = torchvision.models.resnet50(weights="IMAGENET1K_V2").eval()
body = nn.Sequential(*list(net.children())[:-2])          # everything up to layer4 (no pooling, no fc)
fc_conv = nn.Conv2d(2048, 1000, 1)                        # the fc layer as a 1x1 convolution
fc_conv.weight.data = net.fc.weight.data[:, :, None, None].clone()   # (1000, 2048) -> (1000, 2048, 1, 1)
fc_conv.bias.data = net.fc.bias.data.clone()

def norm(x):
    # ImageNet normalisation of a (3, H, W) image, channel by channel
    mean = torch.tensor(torchkit.IMAGENET_MEAN)[:, None, None]    # (3, 1, 1)
    std = torch.tensor(torchkit.IMAGENET_STD)[:, None, None]      # (3, 1, 1)
    return (x - mean) / std

# sanity check on one 224 px image: mean of the score map == the classifier's logits
x224 = norm(pet_val[0][0])[None]
with torch.no_grad():
    a = net(x224); b = fc_conv(body(x224)).mean((2, 3))
print("max |classifier - mean(score map)| =", float((a - b).abs().max()))

# a 448 x 704 canvas with three pets (from the validation set) on a textured background
rng = np.random.default_rng(3)
canvas = np.clip(0.55 + 0.08 * rng.normal(size=(448, 704, 1)) + np.linspace(-.1, .1, 704)[None, :, None], 0, 1).repeat(3, 2)
# one cat and two dogs (cat breeds have file names with a capital first letter, dog breeds a small one)
cat_idx = None
dog_candidates = []
for i in range(len(pet_val)):
    first_letter = pet_val.image_path(i).stem[0]
    if cat_idx is None and first_letter.isupper():
        cat_idx = i                                       # the first cat
    if first_letter.islower():
        dog_candidates.append(i)
dog_idx = dog_candidates[5:7]                             # the 6th and 7th dog
for i, (y, x, s) in zip([cat_idx] + dog_idx, [(40, 30, 230), (190, 300, 230), (60, 500, 190)]):
    im = Image.open(pet_val.image_path(i)).convert("RGB"); r = s / max(im.size)
    im = np.asarray(im.resize((int(im.width * r), int(im.height * r)))) / 255.0
    canvas[y:y + im.shape[0], x:x + im.shape[1]] = im
xc = norm(torch.tensor(canvas, dtype=torch.float32).permute(2, 0, 1))[None]
with torch.no_grad():
    body(xc)                                              # warm-up (memory allocation)
t0 = time.perf_counter()
with torch.no_grad():
    prob = fc_conv(body(xc)).softmax(1)[0]                # (1000, 14, 22): a class distribution per position
t_fc = time.perf_counter() - t0
dog_map = prob[151:269].sum(0).numpy(); cat_map = prob[281:286].sum(0).numpy()
print(f"score map {tuple(prob.shape)} in ONE forward pass: {t_fc:.2f} s  "
      f"(compute = {448 * 704 / 224 ** 2:.1f} crops' worth of 224 x 224)")

In [ ]:
# The same thing done the slow way: crop 224 x 224 windows with stride 64 and classify each
wins = []                                                 # top-left corner (y, x) of every window, row by row
for y in range(0, 448 - 223, 64):
    for x in range(0, 704 - 223, 64):
        wins.append((y, x))
crop_list = []
for y, x in wins:
    crop_list.append(xc[0, :, y:y + 224, x:x + 224])      # (3, 224, 224)
crops = torch.stack(crop_list)                            # (number of windows, 3, 224, 224)
t0 = time.perf_counter()
with torch.no_grad():
    pw = net(crops).softmax(1)
t_sw = time.perf_counter() - t0
ny, nx = len(range(0, 448 - 223, 64)), len(range(0, 704 - 223, 64))
sw_dog = pw[:, 151:269].sum(1).reshape(ny, nx).numpy(); sw_cat = pw[:, 281:286].sum(1).reshape(ny, nx).numpy()

fig, axes = plt.subplots(2, 3, figsize=(16, 7.8))
axes[0, 0].imshow(canvas); axes[0, 0].set_title("input canvas 448 x 704"); axes[0, 0].axis("off")
viz.heatmap_overlay(axes[0, 1], canvas, dog_map, alpha=0.55); axes[0, 1].set_title(f"fully conv.: P(dog)  [1 pass, {t_fc:.1f} s]")
viz.heatmap_overlay(axes[0, 2], canvas, cat_map, alpha=0.55); axes[0, 2].set_title("fully conv.: P(cat)  (stride 32)")
axes[1, 0].imshow(canvas); axes[1, 0].axis("off"); axes[1, 0].set_title(f"{len(wins)} sliding windows (stride 64)")
for (y, x) in wins[::5]:
    axes[1, 0].add_patch(Rectangle((x, y), 224, 224, fill=False, ec=ORANGE, lw=1, alpha=0.7))
for ax, m, nm in [(axes[1, 1], sw_dog, "dog"), (axes[1, 2], sw_cat, "cat")]:
    ax.imshow(canvas); ax.imshow(m, cmap="jet", alpha=0.55, vmin=0, vmax=1, extent=(112, 112 + 64 * nx, 112 + 64 * ny, 112))
    ax.set_xlim(0, 704); ax.set_ylim(448, 0); ax.axis("off")
    ax.set_title(f"sliding window: P({nm})  [{len(wins)} crops, {t_sw:.1f} s]")
plt.tight_layout(); plt.show()

Both maps light up on the animals - but the fully-convolutional one is computed in a single
pass and at a finer stride (32 px instead of 64). This is the seed of every one-stage
detector: *a convnet's feature map is already a grid of window descriptors*. What is missing
is (a) a box that fits the object instead of a fixed window, and (b) a way to handle very
different object sizes. Region proposals were the first answer to both.

### Region proposals: Selective Search in 40 lines

R-CNN did not slide windows; it asked a bottom-up grouping algorithm for ~2000 "object-like"
regions. **Selective Search** over-segments the image into superpixels, then repeatedly merges
the two most similar *neighbouring* regions (similar colour, small, filling their joint
bounding box well) and emits the bounding box of every region it ever forms. Objects appear
at some level of the hierarchy. OpenCV-contrib ships the real thing
(`cv2.ximgproc.segmentation`); we fall back to `day2.mini_selective_search`, a NumPy version of
the same algorithm, when it is not installed.

In [ ]:
try:
    import cv2
    ss_impl = "OpenCV ximgproc"
    def selective_search(img):
        ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
        ss.setBaseImage((img * 255).astype(np.uint8)[..., ::-1]); ss.switchToSelectiveSearchFast()
        r = ss.process()[:500].astype(float)                   # x, y, w, h
        return boxes.xywh_to_xyxy(r), None
except (ImportError, AttributeError):
    ss_impl = "cvcourse.day2.mini_selective_search (OpenCV-contrib not installed)"
    def selective_search(img):
        props, seg, _ = day2.mini_selective_search(img, n_segments=120)
        return props, seg
print("Selective Search implementation:", ss_impl)

pi = 40
im = Image.open(pet_val.image_path(pi)).convert("RGB"); r = 192 / max(im.size)
ss_img = np.asarray(im.resize((int(im.width * r), int(im.height * r)))) / 255.0
ss_gt = pet_val[pi][1]["boxes"].numpy() / 224 * np.array([ss_img.shape[1], ss_img.shape[0]] * 2)
t0 = time.perf_counter(); props, seg = selective_search(ss_img); t_ss = time.perf_counter() - t0
best = boxes.box_iou(ss_gt, props)[0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
if seg is not None:
    axes[0].imshow(seg % 20, cmap="tab20"); axes[0].set_title(f"initial superpixels ({seg.max()})")
else:
    axes[0].imshow(ss_img); axes[0].set_title("input")
axes[1].imshow(ss_img); axes[1].set_title(f"{len(props)} proposals (every 3rd drawn), {t_ss:.1f} s")
for b in props[::3]:
    axes[1].add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec=viz.color(int(b.sum()) % 10), lw=0.6))
top = np.argsort(-best)[:3]
axes[2].imshow(ss_img); axes[2].set_title(f"best proposals for the head: IoU {best[top[0]]:.2f}, {best[top[1]]:.2f}, {best[top[2]]:.2f}")
viz.draw_boxes(axes[2], ss_gt, edgecolor=GREEN, lw=3)
viz.draw_boxes(axes[2], props[top], edgecolor=RED, lw=1.5)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print(f"proposal recall at IoU 0.5 for the head box: {(best >= 0.5).any()}  (R-CNN then ran the CNN on each proposal...)")

Look at the middle and right panels: the proposals know nothing about classes, most of them cover
grass, and only a few fit the head with IoU >= 0.5. Hundreds to thousands of proposals per image
are needed so that every object gets at least one good fit, and R-CNN classified each one with a
full CNN pass: that is where its 47 s per image went.

## 7 · Non-maximum suppression

A good detector fires *several* times on each object (neighbouring windows, anchors or grid
cells all see it). **Greedy NMS** keeps the best and removes its look-alikes:

1. take the highest-scoring remaining box, **keep** it;
2. **remove** every remaining box with IoU $\ge$ threshold (typically 0.5) to it;
3. repeat until no boxes remain.

It is a hand-written post-processing step, not learned - remember that, it comes back in D2-04.

In [ ]:
# Candidate detections around the objects of a shapes image
nms_img, nms_gt, nms_lab, _ = shapes.make_sample(np.random.default_rng(16), 160, max_objects=3, min_r=16, max_r=28)
rng = np.random.default_rng(2)
cand, cand_s = [], []
for g in nms_gt:
    for k in range(5):
        cand.append(g + rng.normal(0, 4, 4)); cand_s.append(rng.uniform(0.5, 0.97))
cand = np.array(cand); cand_s = np.array(cand_s)
print(f"{len(cand)} candidates for {len(nms_gt)} objects")

### Exercise 6 · greedy NMS

`nms(bxs, scores, iou_threshold)` takes `bxs (N, 4)` in xyxy and `scores (N,)` and returns an
integer array with the indices (into `bxs`) of the kept boxes, highest score first. A remaining
box is removed when its IoU with the kept box is $\ge$ `iou_threshold`. Use `boxes.box_iou`.

<details class="api-ref" data-api="torchvision.ops.nms,np.array_equal,np.sort"><summary><b>Functions used here</b> (click to open): <code>torchvision.ops.nms</code> · <code>np.array_equal</code> · <code>np.sort</code></summary>
<p><b><code>torchvision.ops.nms(boxes, scores, iou_threshold)</code></b><br>Greedy non-maximum suppression: repeatedly keeps the highest-scoring remaining box and discards every box whose IoU with it is above iou_threshold. boxes (N, 4) float xyxy, scores (N,); returns the int64 indices of the kept boxes, highest score first.</p><pre>&gt;&gt;&gt; boxes = torch.tensor([[0., 0., 10., 10.],         # 0
...                       [1., 0., 11., 10.],         # 1: almost box 0 (IoU 90 / 110 = 0.82)
...                       [20., 20., 30., 30.]])      # 2: somewhere else
&gt;&gt;&gt; scores = torch.tensor([0.6, 0.9, 0.7])
&gt;&gt;&gt; torchvision.ops.nms(boxes, scores, 0.5)           # 1 is kept and removes 0
tensor([1, 2])</pre><p><i>Watch out:</i> It ignores classes: a dog box can suppress an overlapping cat box. batched_nms runs it per class.</p><p>See also: <code>torchvision.ops.batched_nms</code>, <code>torchvision.ops.box_iou</code></p><p><b><code>np.array_equal(a, b)</code></b><br>True if the two arrays have the same shape and exactly the same elements, as one Python bool (a == b gives an array of element-wise answers instead). For floats with rounding errors use np.allclose.</p><pre>&gt;&gt;&gt; np.array_equal(np.array([1, 2, 3]), [1, 2, 3])
True
&gt;&gt;&gt; np.array_equal(np.array([1, 2]), np.array([1, 2, 3]))     # other shape: False, no error
False
&gt;&gt;&gt; np.array([1, 2, 3]) == np.array([1, 0, 3])
array([ True, False,  True])</pre><p>See also: <code>np.allclose</code>, <code>torch.equal</code></p><p><b><code>np.sort(a, axis=-1)</code></b><br>Returns a sorted copy, smallest first ([::-1] reverses it). D2-01 sorts the indices kept by two NMS implementations before comparing them, so that only the set of kept boxes counts, not the order.</p><pre>&gt;&gt;&gt; a = np.array([3, 1, 2])
&gt;&gt;&gt; np.sort(a), a                    # a itself is unchanged
(array([1, 2, 3]), array([3, 1, 2]))
&gt;&gt;&gt; np.sort(a)[::-1]
array([3, 2, 1])</pre><p><i>Watch out:</i> a.sort() sorts in place and returns None; np.argsort(a) gives the order (indices) instead of the values.</p><p>See also: <code>np.argsort</code>, <code>torch.argsort</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
def nms(bxs, scores, iou_threshold=0.5):
    '''bxs (N, 4) in xyxy, scores (N,) -> integer array with the indices of the kept boxes, highest score first.'''
    # 1. the boxes still in the race, as indices into bxs, highest score first
    order = np.argsort(-scores)                      # (N,)
    keep = []
    # 2. repeat until no box is left
    while len(order) > 0:
        # YOUR CODE: best, the first index of order (the highest score left); append int(best) to keep
        # YOUR CODE: rest, all the other indices of order (order[1:])
        # YOUR CODE: ious, the IoU of box best with each box of rest -> (len(rest),)
        #            (boxes.box_iou(bxs[best][None], bxs[rest]) is a (1, len(rest)) matrix: take its row 0)
        # YOUR CODE: survives, True for the boxes of rest whose IoU with best is < iou_threshold
        # YOUR CODE: order = the indices of rest that survive (rest[survives])
        raise NotImplementedError
    return np.array(keep)

tests.check_nms(nms)

In [ ]:
# Greedy NMS step by step (reference implementation, recording every iteration)
order = list(np.argsort(-cand_s))        # candidates still in the race, highest score first
kept = []
snapshots = []                           # after every step: (kept so far, removed in this step, still waiting)
while order:
    best = order.pop(0)                  # the highest score left ...
    kept.append(best)                    # ... is kept
    if order:
        ious = boxes.box_iou(cand[best][None], cand[order])[0]    # its IoU with every waiting candidate
    else:
        ious = np.array([])
    gone = []                            # IoU >= 0.5 with the kept box: removed
    waiting = []                         # the others stay in the race
    for other, iou in zip(order, ious):
        if iou >= 0.5:
            gone.append(other)
        else:
            waiting.append(other)
    order = waiting
    snapshots.append((list(kept), list(gone), list(order)))
fig, axes = plt.subplots(1, len(snapshots) + 1, figsize=(4 * (len(snapshots) + 1), 4.3))
axes[0].imshow(nms_img); axes[0].set_title(f"{len(cand)} candidates")
viz.draw_boxes(axes[0], cand, scores=cand_s, edgecolor=GREY, lw=1)
for ax, (k, gone, rest) in zip(axes[1:], snapshots):
    ax.imshow(nms_img)
    viz.draw_boxes(ax, cand[rest], edgecolor=GREY, lw=1)
    viz.draw_boxes(ax, cand[gone], edgecolor=RED, lw=1.2)
    viz.draw_boxes(ax, cand[k[:-1]], edgecolor=GREEN, lw=2)
    viz.draw_boxes(ax, cand[k[-1:]], scores=cand_s[k[-1:]], edgecolor=GREEN, lw=3.5)
    ax.set_title(f"step {len(k)}: keep 1 (thick), suppress {len(gone)} (red)", fontsize=10)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

Each panel is one pass of the loop: the thick green box is the one just kept (its score is
written on it), the red boxes are the ones it removed, grey boxes still wait, thin green boxes
were kept earlier. One kept box per object: 15 candidates became 3 detections.

### When greedy NMS hurts: crowds, and Soft-NMS

NMS assumes *one object per neighbourhood*. Two people standing close together, or a
cat in front of a dog, can overlap with IoU > 0.5 - and greedy NMS deletes the second
(correct!) detection. **Soft-NMS** (Bodla et al. 2017) does not delete: it *decays* the
score of each overlapping box, $s_i \leftarrow s_i\, e^{-\mathrm{IoU}(M, b_i)^2/\sigma}$ ($M$ = the box
just kept), so heavy duplicates sink to near zero while a moderately overlapping neighbour
survives with a lower score. Recall goes up at no cost in precision. (The original paper also
has a *linear* variant: $s_i \leftarrow s_i\,(1 - \mathrm{IoU})$ only when IoU $\ge$ the threshold.)

### Lab · NMS playground
**Predict first:** is there one IoU threshold for greedy NMS that keeps both people of a crowd
(their boxes overlap with IoU 0.6) *and* still removes every duplicate in an ordinary street scene?
Step through greedy NMS, then Soft-NMS and class-wise NMS, with the lab's "Try this" list
(7 minutes). The *crowd* scene holds the same four boxes as the figure after Exercise 7.

In [ ]:
labs.show("nms-playground")

### Exercise 7 · Soft-NMS (Gaussian)

Repeatedly pick the remaining box with the highest *current* score, record it, then decay
the scores of the others and drop those below `score_threshold`. Return
`(kept indices, their scores at the time they were picked)`: an integer array and a float array
of the same length, in the order the boxes were picked. This is the loop you stepped through in
the lab; its "now" column is `scores` after each step.

<details class="api-ref" data-api="np.exp"><summary><b>Functions you will need</b> (click to open): <code>np.exp</code></summary>
<p><b><code>np.exp(x)</code></b><br>e = 2.718... to the power of every element. Soft-NMS multiplies scores by exp(-IoU^2 / sigma): 1 for no overlap, close to 0 for a heavy duplicate. It also decodes log-encoded box sizes (D2-02).</p><pre>&gt;&gt;&gt; np.exp(np.array([0.0, 1.0, -1.0]))
array([1.        , 2.71828183, 0.36787944])
&gt;&gt;&gt; iou = np.array([0.0, 0.5, 0.9])
&gt;&gt;&gt; np.exp(-iou ** 2 / 0.5)                   # Soft-NMS decay factors
array([1.        , 0.60653066, 0.1978987 ])</pre><p>See also: <code>np.log</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
def soft_nms(bxs, scores, sigma=0.5, score_threshold=0.001):
    '''bxs (N, 4) in xyxy, scores (N,) -> (kept indices, their scores when picked), in the order they were picked.'''
    scores = np.asarray(scores, float).copy()        # (N,) current scores (a copy: the caller's array stays as it is)
    idx = np.arange(len(scores))                     # indices of the boxes still in the race
    keep = []                                        # the picked indices, in order
    kept_scores = []                                 # their scores at the moment they were picked
    while idx.size > 0:
        # 1. YOUR CODE: m, the index in idx with the highest current score
        #               (position = np.argmax(scores[idx]) is a position inside idx; m = idx[position])
        # 2. YOUR CODE: record it: append int(m) to keep and float(scores[m]) to kept_scores
        # 3. YOUR CODE: remove m from idx (keep the entries of idx that are != m); break if idx is now empty
        # 4. YOUR CODE: ious, the IoU of box m with each remaining box bxs[idx] -> (len(idx),)
        # 5. YOUR CODE: decay, np.exp(-ious ** 2 / sigma); multiply scores[idx] by it
        # 6. YOUR CODE: keep in idx only the boxes whose score is still > score_threshold
        raise NotImplementedError
    return np.array(keep), np.array(kept_scores)

tests.check_soft_nms(soft_nms)

In [ ]:
# Two overlapping pedestrians-like objects, each with its own duplicates
crowd_gt = np.array([[30, 20, 90, 150], [44, 22, 104, 152]], float)
print("IoU between the two people:", round(boxes.iou_single(*crowd_gt), 2))
crowd = np.array([crowd_gt[0] + [1, 2, -1, 0], crowd_gt[0] + [3, -3, 4, 1], crowd_gt[1] + [1, 1, 1, 1], crowd_gt[1] + [-4, 2, -3, 0]])
crowd_s = np.array([0.95, 0.80, 0.90, 0.70])
hard = boxes.nms(crowd, crowd_s, 0.5)
soft_k, soft_s = boxes.soft_nms(crowd, crowd_s, sigma=0.5)
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4), gridspec_kw=dict(width_ratios=[1, 1, 1, 1.3]))
for ax, title, k, s in [(axes[0], "candidates", np.arange(4), crowd_s), (axes[1], f"greedy NMS @0.5: {len(hard)} kept", hard, crowd_s[hard]),
                        (axes[2], "Soft-NMS: all kept, re-scored", soft_k, soft_s)]:
    ax.set_xlim(0, 150); ax.set_ylim(170, 0); ax.set_aspect("equal"); ax.set_title(title, fontsize=10)
    viz.draw_boxes(ax, crowd_gt, edgecolor=GREEN, lw=4, alpha=0.3)
    for i, sc in zip(k, s):
        b = crowd[i]; c = BLUE if i < 2 else ORANGE
        ax.add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec=c, lw=1.8))
        ax.text(b[0] + 2 if i < 2 else b[2] - 2, b[3] - 6 - 14 * (i % 2), f"#{i} {sc:.2f}", color="white", fontsize=8,
                ha="left" if i < 2 else "right", bbox=dict(fc=c, ec="none", pad=1))
x = np.arange(4)
axes[3].bar(x - 0.2, crowd_s, 0.4, color=GREY, label="before")
after = np.zeros(4); after[soft_k] = soft_s
axes[3].bar(x + 0.2, after, 0.4, color=BLUE, label="after Soft-NMS")
axes[3].set_xticks(x); axes[3].set_xticklabels([f"#{i}\n{'person A' if i < 2 else 'person B'}" for i in x])
axes[3].legend(); axes[3].set_title("scores (duplicates sink, person B survives)", fontsize=10)
plt.tight_layout(); plt.show()

### Class-wise NMS

A circle and its bounding square can overlap perfectly and both be correct if they are
*different classes* (person on a horse!). Detectors therefore run NMS **per class**. The
vectorised trick used by `torchvision.ops.batched_nms`: shift every box by
`label * (max coordinate + 1)` so that boxes of different classes can never overlap, then run
one ordinary NMS. (The lab's *person + horse* scene shows the same thing on a realistic layout.)

<details class="api-ref" data-api="torchvision.ops.batched_nms"><summary><b>Functions used here</b> (click to open): <code>torchvision.ops.batched_nms</code></summary>
<p><b><code>torchvision.ops.batched_nms(boxes, scores, idxs, iou_threshold)</code></b><br>NMS run separately per category: a box can only suppress boxes with the same value in idxs (the class labels, an int tensor (N,)). Same inputs and output as nms otherwise: kept indices, highest score first. The label-offset trick shown in D2-01 is how torchvision does it for small inputs.</p><pre>&gt;&gt;&gt; boxes = torch.tensor([[0., 0., 10., 10.], [1., 0., 11., 10.], [20., 20., 30., 30.]])
&gt;&gt;&gt; scores = torch.tensor([0.6, 0.9, 0.7])
&gt;&gt;&gt; torchvision.ops.nms(boxes, scores, 0.5)                                   # class-agnostic
tensor([1, 2])
&gt;&gt;&gt; torchvision.ops.batched_nms(boxes, scores, torch.tensor([0, 1, 0]), 0.5)  # 0 and 1: other classes
tensor([1, 2, 0])</pre><p>See also: <code>torchvision.ops.nms</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Tensor.tolist"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.tolist</code></summary>
<ul><li><code>x.tolist()</code> &nbsp; Converts a tensor into (nested) Python lists of plain Python numbers, for printing a few values readably. A 0-dimensional tensor gives a single number, like .item().</li></ul></details>

In [ ]:
cw_boxes = np.array([[20, 20, 80, 80], [24, 22, 82, 84], [22, 18, 78, 82], [90, 30, 140, 90]], float)
cw_scores = np.array([0.9, 0.85, 0.6, 0.7]); cw_labels = np.array([0, 1, 0, 1])
agnostic = boxes.nms(cw_boxes, cw_scores, 0.5)
classwise = torchvision.ops.batched_nms(torch.tensor(cw_boxes), torch.tensor(cw_scores), torch.tensor(cw_labels), 0.5).numpy()
offset = (cw_boxes.max() + 1) * cw_labels[:, None]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, title, k, bx in [(axes[0], f"class-agnostic NMS: keeps {sorted(agnostic.tolist())}", agnostic, cw_boxes),
                         (axes[1], f"class-wise NMS: keeps {sorted(classwise.tolist())}", classwise, cw_boxes),
                         (axes[2], "the trick: offset boxes by label", np.arange(4), cw_boxes + offset)]:
    for i in k:
        b = bx[i]; c = viz.color(cw_labels[i])
        ax.add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec=c, lw=2))
        ax.text(b[0], b[1] - 3 - 9 * (i == 2) + 70 * (i == 1), f"#{i} {shapes.CLASS_NAMES[cw_labels[i]]} {cw_scores[i]:.2f}", color=c, fontsize=8)
    ax.set_xlim(0, 300 if ax is axes[2] else 150); ax.set_ylim(260 if ax is axes[2] else 110, 0); ax.set_aspect("equal"); ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

### Hard-negative mining

A detector looks at thousands of candidate windows per image and only a handful contain
objects. If you train on all of them, the loss is dominated by **easy negatives** (empty
background the classifier already rejects); if you sample negatives uniformly, you mostly
pick easy ones too. **Hard-negative mining** (R-CNN, SSD) trains on the negatives with the
*highest loss*: typically all positives + the top-scoring negatives at a 1:3 ratio. Below, a
toy "detector" that fires on colourful windows scores every 32 x 32 window of a shapes image.

<details class="api-ref" data-api="Tensor.min,np.log"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.min</code> · <code>np.log</code></summary>
<ul><li><code>x.min() / x.min(dim)</code> &nbsp; x.min() is the smallest element (a 0-dimensional tensor); x.min(dim) works along one dimension and returns two tensors, the minimum values and their positions. NumPy arrays have the same method: arr.min() and arr.min(axis=...), which returns only the values.</li><li><code>np.log(x)</code> &nbsp; The natural logarithm (base e) of every element. ln C is the cross-entropy loss of a classifier that guesses uniformly among C classes: ln 37 = 3.61.</li></ul></details>

In [ ]:
hn_img, hn_gt, _, _ = shapes.make_sample(np.random.default_rng(4), 160, max_objects=4, min_r=12, max_r=20)
sat = hn_img.max(-1) - hn_img.min(-1)                          # (160, 160) how colourful every pixel is
win_list = []                                                  # every 32 x 32 window (stride 4), xyxy
mean_sat = []                                                  # the mean colourfulness inside it
for y in range(0, 129, 4):
    for x in range(0, 129, 4):
        win_list.append([x, y, x + 32, y + 32])
        mean_sat.append(sat[y:y + 32, x:x + 32].mean())
wins = np.array(win_list, float)                               # (1089, 4)
score = np.array(mean_sat)                                     # (1089,)
score = 1 / (1 + np.exp(-30 * (score - 0.25)))                 # toy classifier: P(object | colourful window)
is_pos = boxes.box_iou(wins, hn_gt).max(1) >= 0.5
neg_loss = -np.log(1 - score[~is_pos] + 1e-6)                 # BCE of each negative
n_hard = 3 * is_pos.sum()
hard_idx = np.where(~is_pos)[0][np.argsort(-neg_loss)[:n_hard]]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), gridspec_kw=dict(width_ratios=[1, 1, 1.2]))
axes[0].imshow(hn_img); axes[0].set_title(f"{len(wins)} windows: {is_pos.sum()} positive (IoU >= 0.5)")
viz.draw_boxes(axes[0], wins[is_pos], edgecolor=GREEN, lw=1.5)
axes[1].imshow(hn_img); axes[1].set_title(f"the {n_hard} hardest negatives (1:3 ratio)")
viz.draw_boxes(axes[1], wins[hard_idx], edgecolor=RED, lw=1)
for ax in axes[:2]: ax.axis("off")
axes[2].hist(neg_loss, bins=40, color=GREY, log=True); axes[2].axvline(np.sort(neg_loss)[-n_hard], color=RED, ls="--", label="hard-negative cut")
axes[2].set_xlabel("loss of a negative window"); axes[2].set_ylabel("count (log)"); axes[2].legend()
axes[2].set_title(f"{(~is_pos).sum()} negatives: {100 * (neg_loss < 0.05).mean():.0f}% are easy (loss < 0.05)")
plt.tight_layout(); plt.show()
print(f"share of total negative loss carried by the {n_hard} hard negatives: {np.sort(neg_loss)[-n_hard:].sum() / neg_loss.sum():.0%}")

The hard negatives are exactly the windows that *half* cover a shape - the confusing cases
the detector must learn to reject. Focal loss (D2-03) replaces this hand-made sampling with a
smooth re-weighting of every example, which is why one-stage detectors could finally train
on all anchors.

**NMS-free detectors.** DETR (2020) and YOLOv10 (2024) are trained with a *one-to-one*
assignment between predictions and objects (Hungarian matching, D2-04), so the network itself
learns not to produce duplicates and the NMS step - with its threshold and its latency - disappears.

## 8 · Summary

* A detector outputs a variable-length set of `(box, class, score)`. Boxes come in xyxy / xywh /
  cxcywh, in pixels or normalised - know which one every tool expects.
* IoU = intersection / union; vectorise it with broadcasting. IoU falls to 0.5 after a shift of
  one third of the box width, and the same pixel error is much worse for small objects.
* $1-\mathrm{IoU}$ has no gradient without overlap; GIoU (enclosing box), DIoU (centre distance)
  and CIoU (+ aspect ratio) fix that and are the box losses of DETR and YOLO.
* Evaluation: sort by score, greedy matching (duplicates are FPs), PR curve, AP = area under the
  monotone envelope, mAP = mean over classes; COCO averages over IoU 0.5:0.95 and reports S / M / L.
* The lineage is a chain of removed bottlenecks; the fully-convolutional trick turns a classifier
  into a dense detector in one pass.
* NMS removes duplicates greedily; Soft-NMS decays instead of deleting (crowds); NMS is run per class;
  hard-negative mining keeps the loss from drowning in easy background.

### Self-check
1. Box `(10, 20, 50, 80)` in xyxy: what are its xywh and cxcywh forms? Its area?
2. Two 40 x 40 boxes, one shifted 10 px right and 10 px down. IoU? (Draw it.)
3. Why is $1-\mathrm{IoU}$ useless as a loss for a prediction that does not overlap the GT? What does GIoU add?
4. A detector finds every object but also produces one duplicate per object, with a lower score. What happens to its AP@0.5?
5. Why can a mAP@0.5 of 0.95 coexist with a mAP@[.5:.95] of 0.55?
6. In a crowded scene, what goes wrong with an NMS threshold of 0.3? With 0.9?

### Go deeper
* Girshick et al., *Rich feature hierarchies for accurate object detection and semantic segmentation* (R-CNN, 2014) and Girshick, *Fast R-CNN* (2015).
* Uijlings et al., *Selective Search for Object Recognition* (IJCV 2013).
* Rezatofighi et al., *Generalized Intersection over Union* (CVPR 2019); Zheng et al., *Distance-IoU Loss* (AAAI 2020, DIoU and CIoU).
* COCO detection evaluation documentation (cocodataset.org/#detection-eval) - the exact protocol behind mAP@[.5:.95].
* Padilla et al., *A Comparative Analysis of Object Detection Metrics with a Companion Open-Source Toolkit* (Electronics 2021).
* Bodla et al., *Soft-NMS - Improving Object Detection With One Line of Code* (ICCV 2017).